# Team ratings + game script — Tier 1 study (NFL)

**Read-only statistical study. This notebook changes no model, no pickle, no feature
list and no site code.** It fits throwaway estimators inside itself to answer four
questions, and every result is reported as *the change over the repo's current
feature set*, never over nothing.

* Baseline = `training.QUANTILE_FEATURE_COLUMNS` (the 14 lagged rolling columns the props
  route actually serves). `FORWARD_FEATURE_COLUMNS` (38) is the forward set and is **not**
  the baseline: it was narrowed to the served 14 on 2026-10-15 because the other 24 are not
  produced by `player_usage.build_features_for_player` and were being zero-filled.
* Harness, metrics and gates are the repo's own: `evaluate.walk_forward.walk_forward_quantile`,
  `calibration_report` (10 bins, **tolerance 0.05, n>=100 — not lowered**), `calibration_passes`,
  `models.prop_probability.p_over_from_quantiles`, `models.player_props.fit_yardage_quantile_models`,
  `models.receptions_zinb`, `models.qb_passing_td`. No parallel evaluation is built.
* Validation seasons 2019–2025 (`scripts/train_quantile_props.py`'s own default).

**Pre-stated, before any result was seen** (rigor rule: these are not retuned afterwards):
seasons 2017–2025 for fitting, 2019–2025 for validation; qualifying-player threshold
**>= 6 games in the season**; garbage-time exclusion **surrogate P(win) outside [0.10, 0.90]**,
reported both with and without; primary quantile levels **q10/q50/q90**; paired bootstrap
**1,000 reps, resampled by player**.

**Two honest disclosures up front.**

1. `pbp_{2017..2026}.parquet` has **no `pwin`/`win_probability` column**, so the prompt's
   "exclude plays outside 10–90% win probability" cannot be applied literally. A causal
   surrogate is fitted instead (Section 1) and the rule is stated there.
2. `receptions_zinb._derive_features` hard-codes its own 9 columns, so a ratings arm cannot
   reach it without editing the module. To keep the ZINB *likelihood, line function and gate
   arithmetic* as the repo's own while still measuring the question, the notebook **monkeypatches
   `_derive_features` in memory only** (`Part A cell 6`). No file on disk is touched. The
   receptions result is reported with that caveat repeated.

In [ ]:
import json, math, os, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# Resolve the repo root: an explicit override, or the nearest ancestor of the current
# directory that actually carries `src/nfl_predictor/models/training.py`. The notebook is
# committed under `notebooks/`, so `Path.cwd()` alone would resolve to `notebooks/` and import
# the wrong `nfl_predictor` (the editable install points at the main checkout, not this one).
def _repo_root():
    if os.environ.get("NFL_STUDY_REPO"):
        return Path(os.environ["NFL_STUDY_REPO"])
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand / "src" / "nfl_predictor" / "models" / "training.py").exists():
            return cand
    return here
REPO = _repo_root()
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "scripts"))

# The cached nflverse pull is gitignored and lives in the primary checkout. Absolute path,
# so the notebook never writes inside data/cache/ of the worktree it was committed from.
CACHE = Path(os.environ.get("NFL_STUDY_CACHE", "/Users/sigey/Documents/Projects.nosync/NFL_Predictor/data/cache")) / "nflverse"
STUDY_CACHE = REPO / "data" / "cache" / "study"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from nfl_predictor.models.training import (
    QUANTILE_FEATURE_COLUMNS, FORWARD_FEATURE_COLUMNS, MARKET_POSITIONS,
    MARKET_LABEL_COLUMN, COUNT_MARKETS,
)
from nfl_predictor.evaluate.walk_forward import (
    walk_forward_quantile, calibration_report, calibration_passes, QUANTILE_MARKETS,
)
from nfl_predictor.models.prop_probability import p_over_from_quantiles
from nfl_predictor.models import player_props, receptions_zinb, qb_passing_td
from nfl_predictor.models.receptions_zinb import RECEPTIONS_FEATURE_COLUMNS as ZINB_COLS
import nfl_predictor.evaluate.walk_forward as WF

# ---- pre-stated study constants (see the header; not retuned after seeing results) ----
FIT_SEASONS    = list(range(2017, 2026))
VALID_SEASONS  = list(range(2019, 2026))
GAMES_MIN      = 6                 # qualifying player: >= 6 games that season
GARBAGE_LO, GARBAGE_HI = 0.10, 0.90
QUANTILES      = [0.1, 0.5, 0.9]
BOOTSTRAP_REPS = 1000
RNG            = np.random.default_rng(20261017)

PRICED_POSITIONS = ["QB", "RB", "WR", "TE"]
RATING_STATS = ["pass_off", "pass_def", "rush_off", "rush_def"]

# Part A markets. The first four are the repo's own QUANTILE_MARKETS; the three TD/usage
# markets are registered into the SAME harness dict at run time so the arithmetic cannot
# drift from the repo's (see cell 6). No module file is modified.
EXTRA_MARKETS = {
    "rushing_tds":  ("RB",),          # RB rush TD
    "receiving_tds": ("WR", "TE"),    # WR/TE receiving TD
    "attempts":     ("QB",),          # QB pass attempts (Part B.2)
    "carries":      ("RB",),          # RB carries (Part B.2)
    "targets":      ("WR", "TE"),     # WR/TE targets (Part B.2)
}
EXTRA_LABEL_COLUMN = {
    "rushing_tds": "rushing_tds", "receiving_tds": "receiving_tds",
    "attempts": "attempts", "carries": "carries", "targets": "targets",
}

print("baseline feature set :", len(QUANTILE_FEATURE_COLUMNS), QUANTILE_FEATURE_COLUMNS)
print("forward feature set  :", len(FORWARD_FEATURE_COLUMNS))
print("fit seasons          :", FIT_SEASONS)
print("validation seasons   :", VALID_SEASONS)
print("qualifying threshold :", GAMES_MIN, "games/season")
print("garbage-time rule    : surrogate P(win) outside", [GARBAGE_LO, GARBAGE_HI])
print("quantile levels      :", QUANTILES)
print("bootstrap reps       :", BOOTSTRAP_REPS)

## Section 0 — what data and ratings the repo already has, and the row counts

**Plain English.** The repo already carries more team-level signal than the props models are
allowed to see. Three opponent-defence rolling features (`opp_pass/rush/rec_yds_allowed_roll`),
a team tempo block (`plays_roll`, `pass_share_roll`, `opp_plays_allowed_roll`) and the Elo
power ratings all exist and are causal, but none of them are in the 14-column served baseline —
they were dropped from it on 2026-10-15 precisely because the props route does not build them.
So the honest prior for Part A is **not** "ratings are new information"; it is "ratings may
already be captured by a feature the baseline is not allowed to use". The study frame is
player-game, priced positions only, with players required to appear in >= 6 games. That leaves
43,531 rows over 2017–2025 and drops the 2026 partial season (5 weeks) entirely, which is
stated rather than worked around by loosening the threshold.

In [ ]:
print("=== ratings / rating-like features the repo ALREADY builds ===")
inventory = pd.DataFrame([
    ["Elo/MoV team power ratings", "features/power_ratings.py::compute_pregame_ratings", "game-outcome models only", "no"],
    ["opp_pass/rush/rec_yds_allowed_roll", "features/matchup.py::_opponent_allowed", "forward set only (dropped from served 14 on 2026-10-15)", "no"],
    ["plays_roll / pass_share_roll / opp_plays_allowed_roll / opp_pressure_rate_roll", "features/interactions.py::add_pbp_features", "forward set only", "no"],
    ["target_share / expected_air_yards / target_concentration / expected_plays", "features/interactions.py", "forward set only", "no"],
    ["team EPA efficiency", "data/pbp_agg.py (epa block)", "game-outcome models only", "no"],
    ["14 lagged rolling columns", "features/player_usage.py::build_features_for_player", "THE props baseline", "yes"],
], columns=["feature", "where", "used by", "in baseline"])
display(inventory)

print("\n=== forward-set columns that are NOT in the served baseline (candidates already in the repo) ===")
dropped = [c for c in FORWARD_FEATURE_COLUMNS if c not in QUANTILE_FEATURE_COLUMNS]
print(f"{len(dropped)} dropped:", dropped)

# ---------------------------------------------------------------- feature frame
t0 = time.time()
STUDY_CACHE.mkdir(parents=True, exist_ok=True)
FRAME_PATH = STUDY_CACHE / "feature_frame.parquet"
if FRAME_PATH.exists():
    frame = pd.read_parquet(FRAME_PATH)
    print(f"\nfeature frame: reused cache {FRAME_PATH} ({frame.shape[0]} rows)")
else:
    import train_quantile_props as TQP
    print("\nbuilding the feature frame with the repo's own builder "
          "(scripts/train_quantile_props.py::build_feature_frame)...")
    frame = TQP.build_feature_frame(list(range(2017, 2027)), CACHE, fetch_weather=False)
    frame.to_parquet(FRAME_PATH, index=False)
    print(f"feature frame: built and cached to {FRAME_PATH} ({frame.shape[0]} rows)")
print(f"frame build/reuse took {time.time() - t0:.1f}s; columns={len(frame.columns)}")

# ---------------------------------------------------------------- filters
def qualifying_filter(f, games_min=GAMES_MIN, positions=PRICED_POSITIONS):
    """Priced positions AND >= `games_min` games that season. Returns the filtered frame."""
    f = f[f["position"].isin(positions)].copy()
    played = f.groupby(["season", "player_id"]).size().rename("games").reset_index()
    keep = set(zip(played.loc[played.games >= games_min, "season"],
                   played.loc[played.games >= games_min, "player_id"]))
    mask = [(s, p) in keep for s, p in zip(f["season"], f["player_id"])]
    return f[mask].reset_index(drop=True)

ROWCOUNT_TABLE = []
for label, f in (("all positions", frame),
                 ("priced QB/RB/WR/TE", frame[frame.position.isin(PRICED_POSITIONS)]),
                 (f"+ >= {GAMES_MIN} games/season (ANALYSIS FRAME)", qualifying_filter(frame))):
    per = f.groupby("season").size()
    ROWCOUNT_TABLE.append({"filter": label, "total": len(f),
                           **{str(s): int(per.get(s, 0)) for s in range(2017, 2027)}})
rowcounts = pd.DataFrame(ROWCOUNT_TABLE)
display(rowcounts)

In [ ]:
analysis = qualifying_filter(frame)
analysis = analysis[analysis.season <= 2025]

print(f"\nANALYSIS FRAME: {len(analysis)} player-games, "
      f"{analysis.player_id.nunique()} players, {analysis.season.nunique()} seasons\n")

rows = []
for mkt, poss in {**MARKET_POSITIONS, **EXTRA_MARKETS}.items():
    label = MARKET_LABEL_COLUMN.get(mkt) or EXTRA_LABEL_COLUMN[mkt]
    sub = analysis[analysis.position.isin(poss) & analysis[label].notna()]
    rows.append({
        "market": mkt, "position": "/".join(poss), "label": label, "rows": len(sub),
        "mean": round(sub[label].mean(), 2), "median": round(sub[label].median(), 1),
        "pct_zero": round((sub[label] == 0).mean(), 3),
        "per_season": {str(s): int(n) for s, n in sub.groupby("season").size().items()},
    })
market_rows = pd.DataFrame(rows)
display(market_rows.drop(columns=["per_season"]))
for _, r in market_rows.iterrows():
    print(f"  {r.market:15s} per-season {r.per_season}")

print("\nskew check (justifies log1p / count models over OLS):")
for mkt in ("passing_yards", "rushing_yards", "receiving_yards", "rushing_tds", "passing_tds"):
    sub = analysis[analysis[MARKET_LABEL_COLUMN.get(mkt, EXTRA_LABEL_COLUMN.get(mkt, mkt))].notna()]
    y = sub[mkt]
    print(f"  {mkt:15s} skew={y.skew():6.2f}  kurtosis={y.kurtosis():6.2f}  "
          f"p0={y.min():6.1f} p50={y.median():6.1f} p99={y.quantile(.99):7.1f} max={y.max():7.1f}")

print("\npre-game input coverage on the analysis frame:")
for c in ("spread_line", "total_line", "is_home", "rest_days", "temp_c", "wind_kph",
          "plays_roll", "pass_share_roll", "opp_plays_allowed_roll", "target_share"):
    if c in analysis.columns:
        print(f"  {c:24s} non-null {analysis[c].notna().mean():.3f}")
print("\nNOTE: nflverse weekly spread_line/total_line are CLOSING lines. training.py excludes "
      "them from the baseline for exactly this train/serve skew reason, so every spread/total "
      "result below is an UPPER BOUND: a pre-kickoff snapshot only has the opening line.")

## Section 1 — building the ratings, and proving they are causal

**Plain English.** Four ratings are built per player-game: **pass-off** (how many passing yards
the player's own team generates, *excluding the player*), **pass-def** (how many passing yards the
opponent's defence allows to everyone *else*), **rush-off** and **rush-def** likewise. Each is a
shrunk rolling mean over the team's games **strictly earlier in the same season**, shrunk toward
the team's own **last-season** value so week 1 is not a guess from scratch. The
leave-one-player-out rule is the part that is easy to get wrong. A player's own passing yards sit
inside his team's offensive output, so without removing them the model would predict a player
from a rating that already contains him. The same trap bites the defence side through the
divisional rematch, where the opponent's "yards allowed" history already contains this player's
own production. Both sides are corrected, and the LOPO unit follows the player's role — a QB's
yardage-for is passing, a WR's is receiving, an RB's is rushing — because passing the wrong one
is precisely the silent bug this rule exists to stop. Three tests below then **prove** causality
instead of asserting it.

In [ ]:
# ------------------------------------------------------------------ play-by-play (slim)
print("loading play-by-play (slim columns) ...")
PB = ["season", "week", "game_id", "posteam", "defteam", "home_team", "away_team",
      "home_score", "away_score", "qtr", "pass_attempt", "rush_attempt", "passing_yards",
      "rushing_yards", "complete_pass", "pass_touchdown", "rush_touchdown",
      "passer_player_id", "rusher_player_id", "receiver_player_id", "play_id", "season_type"]
if (STUDY_CACHE / "pbp_slim.parquet").exists():
    pbp = pd.read_parquet(STUDY_CACHE / "pbp_slim.parquet")
else:
    import pyarrow.parquet as _pq
    frames = []
    for s in range(2017, 2027):
        path = CACHE / f"pbp_{s}.parquet"
        avail = set(_pq.ParquetFile(path).schema_arrow.names)
        frames.append(pd.read_parquet(path, columns=[c for c in PB if c in avail])
                      .assign(**{c: np.nan for c in PB if c not in avail}))
    pbp = pd.concat(frames, ignore_index=True)
    pbp.to_parquet(STUDY_CACHE / "pbp_slim.parquet", index=False)
if "season_type" in pbp.columns:
    pbp = pbp[pbp.season_type.astype(str).str.upper() == "REG"]
print(f"pbp: {len(pbp):,} rows, seasons {sorted(pbp.season.unique())}")
print("real win-probability column present?", any(c in pbp.columns for c in ("pwin", "win_probability")))

B = lambda c: pbp[c].fillna(0).astype(bool)
offensive = pbp[B("pass_attempt") | B("rush_attempt")]

# ------------------------------------------------------------------ game frame + repo Elo
game_scores = (pbp.groupby(["season", "week", "game_id"], as_index=False)
               .agg(home_team=("home_team", "first"), away_team=("away_team", "first"),
                    home_score=("home_score", "last"), away_score=("away_score", "last")))

from nfl_predictor.features.power_ratings import compute_pregame_ratings, ELO_SCALE, DEFAULT_HOME_FIELD

def game_elo(schedule: pd.DataFrame) -> pd.DataFrame:
    """The repo's own Elo (`compute_pregame_ratings`) on the game's own frame. That function is
    chronological: the rating it writes on a row reflects only games strictly before it."""
    sched = schedule.sort_values(["season", "week", "game_id"]).copy()
    sched["gameday"] = (pd.to_datetime(sched.season.astype(str) + "-09-01")
                        + pd.to_timedelta(sched.week * 7, unit="D"))
    rated = compute_pregame_ratings(sched[["game_id", "gameday", "home_team", "away_team",
                                           "home_score", "away_score", "season", "week"]])
    return rated[["game_id", "season", "week", "home_pregame_rating", "away_pregame_rating"]]

elo = game_elo(game_scores)
gs = game_scores.merge(elo, on=["game_id", "season", "week"], how="left")

team_games = (offensive.groupby(["season", "week", "game_id", "posteam"], as_index=False)
              .agg(pass_attempts=("pass_attempt", "sum"), rush_attempts=("rush_attempt", "sum"),
                   plays=("play_id", "size"), pass_yards=("passing_yards", "sum"),
                   rush_yards=("rushing_yards", "sum"))
              .rename(columns={"posteam": "team"}))
allowed = (offensive.groupby(["season", "week", "game_id", "defteam"], as_index=False)
           .agg(allowed_pass_yards=("passing_yards", "sum"),
                allowed_rush_yards=("rushing_yards", "sum"),
                pass_attempts_allowed=("pass_attempt", "sum"),
                rush_attempts_allowed=("rush_attempt", "sum"))
           .rename(columns={"defteam": "team"}))
team_games = team_games.merge(allowed, on=["season", "week", "game_id", "team"], how="left")
team_games = team_games.merge(
    gs.assign(home=lambda d: d.home_team, away=lambda d: d.away_team)[
        ["season", "week", "game_id", "home", "away", "home_score", "away_score",
         "home_pregame_rating", "away_pregame_rating"]],
    on=["season", "week", "game_id"], how="left")
team_games["opponent"] = np.where(team_games.team == team_games.home, team_games.away, team_games.home)
team_games["is_home"] = (team_games.team == team_games.home).astype(int)
team_games["points"] = np.where(team_games.is_home == 1, team_games.home_score, team_games.away_score)
team_games["elo_diff"] = np.where(team_games.is_home == 1,
                                  team_games.home_pregame_rating - team_games.away_pregame_rating,
                                  team_games.away_pregame_rating - team_games.home_pregame_rating)
team_games["pwin_pregame"] = 1.0 / (1.0 + 10.0 ** (-team_games.elo_diff / ELO_SCALE))
print(f"\nteam-games {len(team_games):,} rows / {team_games.game_id.nunique():,} games; "
      f"home-favourite rate {float((team_games.loc[team_games.is_home == 1, 'elo_diff'] > 0).mean()):.3f} "
      "(sanity: near the NFL home rate, not 0.50)")
display(team_games.groupby("season").size().rename("team_games").to_frame())

In [ ]:
# ------------------------------------------------------------------ garbage-time: NOT COMPUTABLE
# The prompt asks to "exclude garbage-time plays from rating construction (win probability
# outside 10-90%, state the rule)". `pbp_{2017..2026}.parquet` has NO `pwin` /
# `win_probability` column, so the rule cannot be applied to the real quantity. Everything below
# is the evidence for what happens when a surrogate is built instead -- and why the resulting
# arm is reported as NOT COMPUTABLE rather than shipped as a number.
#
# Two facts are established and printed:
#   1. nflverse pbp's home_score/away_score are the FINAL score (one unique pair per game), so
#      they cannot serve as the live game state.
#   2. A live margin CAN be rebuilt from the touchdown columns (6 pts per touchdown, home and
#      away accumulated separately). Its reconstruction correlates 0.875 with the real final
#      margin, so the state variable itself is sound.
# The failure is in the SURROGATE, not the state. Three formulations were fitted, all on causal
# data (2017 only), and all three saturate:
#   (a) additive quarter term        -> ties in Q1 score P(win) = 0.027, 100% of plays flagged
#   (b) time-weighted margin, all plays, ties coded "not won" -> ties score 0.497 only after
#       dropping ties, but +7 in Q3 still scores 0.997 and 63% of plays are flagged
#   (c) time-weighted margin, one observation per team per HALF -> ties 0.506 (correct), but
#       +7 at the half still scores ~0.99 and 63% of plays are flagged
# The 10-90% band identifies a vanishing fraction of plays under every formulation, so a
# "with garbage time" ratings build would be built from an arbitrary threshold rather than from
# the rule the prompt specifies. It is therefore NOT reported as a result.
#
# CONSEQUENCE, stated plainly: every rating in this notebook is built WITHOUT the garbage-time
# exclusion, and there is no second arm to compare it against. This is a gap in coverage, not a
# null result: the study cannot say whether the exclusion would change the answer.
from sklearn.linear_model import LogisticRegression

# Attach the PRE-GAME Elo diff to every play: game-constant, built from earlier games only.
plays = offensive.merge(
    team_games[["season", "week", "game_id", "team", "elo_diff", "pwin_pregame"]],
    left_on=["season", "week", "game_id", "posteam"], right_on=["season", "week", "game_id", "team"],
    how="left").drop(columns=["team"])

plays["pts"] = (6.0 * plays["pass_touchdown"].fillna(False).astype(float)
                + 6.0 * plays["rush_touchdown"].fillna(False).astype(float))
if "extra_point_result" in plays.columns:
    plays["pts"] += plays["extra_point_result"].eq("made").astype(float)
if "two_point_conv_result" in plays.columns:
    plays["pts"] += 2.0 * plays["two_point_conv_result"].eq("success").astype(float)
plays = plays.sort_values(["season", "game_id", "play_id"])
_is_home = (plays.posteam == plays.home_team).to_numpy()
plays["home_pts"] = np.where(_is_home, plays.pts.to_numpy(float), 0.0)
plays["away_pts"] = np.where(_is_home, 0.0, plays.pts.to_numpy(float))
_gp = plays.groupby(["season", "game_id"])
plays["home_cum"] = _gp["home_pts"].cumsum().to_numpy()
plays["away_cum"] = _gp["away_pts"].cumsum().to_numpy()
plays["margin_home_pov"] = plays.home_cum - plays.away_cum
plays["live_margin"] = np.where(_is_home, plays.margin_home_pov.to_numpy(float),
                                -plays.margin_home_pov.to_numpy(float))
plays["won"] = (plays.live_margin > 0).astype(int)

_end = _gp.tail(1)
print("EVIDENCE 1 -- nflverse pbp carries no win-probability field:")
print("           columns matching pwin/win_probability:",
      [c for c in pbp.columns if "win" in c.lower() and "prob" in c.lower()] or "NONE")
print("EVIDENCE 2 -- a live margin rebuilt from touchdowns is a sound state variable:")
print(f"           corr(reconstructed final differential, real final margin) = "
      f"{float(np.corrcoef(_end.margin_home_pov, _end.home_score - _end.away_score)[0, 1]):.3f}")
print(f"           share of plays with |live margin| >= 14 = "
      f"{float((plays.live_margin.abs() >= 14).mean()):.3f}")
print(f"           share of plays TIED at the moment of the play = "
      f"{float((plays.live_margin == 0).mean()):.3f}   <- a tie must score 0.5, not 0")

plays["half"] = np.where(plays.qtr.fillna(4) <= 2, 1, 2)
plays["frac_remaining"] = np.where(plays.half == 1, 0.5, 0.12)
plays["margin_weighted"] = plays.live_margin * np.sqrt(plays.frac_remaining)
_half = plays.groupby(["season", "game_id", "posteam", "half"]).tail(1).copy()

def _fit_and_flag(train: pd.DataFrame, label: str):
    m = LogisticRegression(max_iter=500).fit(train[["margin_weighted"]], train["won"])
    p_all = m.predict_proba(plays[["margin_weighted"]])[:, 1]
    flag = (p_all < GARBAGE_LO) | (p_all > GARBAGE_HI)
    probe = {s: round(float(m.predict_proba(
        pd.DataFrame({"margin_weighted": [lm * np.sqrt(rem)]}))[0, 1]), 3)
        for s, lm, rem in [("tied H1", 0, 0.5), ("+7 H2", 7, 0.12), ("+14 H2", 14, 0.12)]}
    print(f"  {label:34s} coef={float(m.coef_[0][0]):7.3f}  flagged={flag.mean():.4f}  "
          f"P(tied H1)={probe['tied H1']}  P(+7 H2)={probe['+7 H2']}  P(+14 H2)={probe['+14 H2']}")

print("\nEVIDENCE 3 -- every causal surrogate saturates, so the 10-90% band is unidentified:")
_a = plays[(plays.season == 2017) & plays.qtr.notna()].copy()
_fit_and_flag(_a, "(a) all plays, ties = not won")
_b = _a[_a.live_margin != 0]
_fit_and_flag(_b, "(b) all plays, ties dropped")
_c = _half[(_half.season == 2017) & (_half.live_margin != 0)]
_fit_and_flag(_c, "(c) one obs per team per half")

print("\nVERDICT: garbage-time exclusion is NOT COMPUTABLE on this dataset. The 10-90% rule needs")
print("a win-probability field that nflverse does not publish, and every surrogate built from")
print("play-by-point state flags 63-100% of plays, so any threshold would be arbitrary.")
print("Ratings are therefore built WITHOUT the exclusion, and there is no 'with' arm to compare.")
print("Reported in the PR body as refused, not as a null.")

In [ ]:
# ------------------------------------------------------------------ rating construction
PLAYER_COLS = ["season", "week", "player_id", "recent_team", "opponent_team", "position",
               "passing_yards", "rushing_yards", "receiving_yards", "passing_tds", "rushing_tds",
               "receiving_tds", "attempts", "carries", "targets", "receptions"]
PG = analysis[PLAYER_COLS].copy()

TEAM_CUM = ("pass_attempts", "rush_attempts", "plays", "pass_yards", "rush_yards",
            "allowed_pass_yards", "allowed_rush_yards", "pass_attempts_allowed",
            "rush_attempts_allowed")
#: Every team-level rating that gets a cross-sectional (within-season) scaling.
RATING_STATS_TM = ["tm_pass_off", "tm_rush_off", "tm_pass_def", "tm_rush_def",
                   "tm_pass_att_off", "tm_rush_att_off", "tm_pass_att_def", "tm_rush_att_def"]
SHRINK_K_PROVISIONAL = 6.0   # replaced by the measured value in cell 1a; kept so the frame builds first


def build_team_games(src: pd.DataFrame | None = None) -> pd.DataFrame:
    """Team-game aggregates from the play-by-play.

    There is no garbage-time filter here, deliberately: the 10-90% win-probability rule the
    prompt asks for is not computable on this dataset (see the previous cell), so filtering on
    an arbitrary threshold would be inventing an exclusion, not applying the stated one."""
    s = (src if src is not None else plays)
    own = (s.groupby(["season", "week", "game_id", "posteam"], as_index=False)
           .agg(pass_attempts=("pass_attempt", "sum"), rush_attempts=("rush_attempt", "sum"),
                plays=("play_id", "size"), pass_yards=("passing_yards", "sum"),
                rush_yards=("rushing_yards", "sum")))
    alw = (s.groupby(["season", "week", "game_id", "defteam"], as_index=False)
           .agg(allowed_pass_yards=("passing_yards", "sum"),
                allowed_rush_yards=("rushing_yards", "sum"),
                pass_attempts_allowed=("pass_attempt", "sum"),
                rush_attempts_allowed=("rush_attempt", "sum")))
    out = own.rename(columns={"posteam": "team"}).merge(
        alw.rename(columns={"defteam": "team"}), on=["season", "week", "game_id", "team"], how="left")
    out = out.merge(team_games[["season", "week", "game_id", "team", "opponent", "is_home",
                                "pwin_pregame", "elo_diff", "points"]],
                    on=["season", "week", "game_id", "team"], how="left")
    # float64, deliberately. nflverse ships these columns as float32, and a float32 running
    # total over a season picks up rounding error that grows with n. It is invisible in the
    # ratings themselves but it made leakage test 2 report a spurious ~0.1 change: scaling one
    # week by 1e6 perturbed the float32 cumsum of every later week by a unit or two. The test
    # was not wrong about leakage, it was measuring float32 noise. Casting once here makes the
    # whole study's arithmetic deterministic AND keeps the leakage assertions at exactly 0.0
    # rather than being loosened into a tolerance.
    out[list(TEAM_CUM)] = out[list(TEAM_CUM)].astype("float64")
    return out.sort_values(["season", "team", "week"]).reset_index(drop=True)


def _last_season_prior(tg: pd.DataFrame, col: str) -> np.ndarray:
    """Team T's full-season mean of `col` in season S-1, aligned to tg's rows. Causal: last
    season is complete before this one starts. Falls back to that season's league mean, then 0.

    THE JOIN KEY IS THE WHOLE POINT. A season-S game must read the season-(S-1) mean, so the
    prior table is keyed by `season + 1` -- "the season this value may be used in" -- and matched
    against the game's own season. The first version wrote `season - 1` and matched that against
    the game's season, which silently handed every game the NEXT season's mean: a one-season
    lookahead hiding in the shrinkage prior. Leakage test 1 is what found it; without that test
    it would have shipped as a quietly optimistic rating for every early-week row.
    """
    tm = tg.groupby(["season", "team"], as_index=False)[col].mean()
    league = tg.groupby("season")[col].mean()
    prior_table = tm.assign(usable_from=tm["season"] + 1)[["usable_from", "team", col]] \
                     .rename(columns={col: "_prev"})
    key = tg[["season", "team"]].merge(prior_table, left_on=["season", "team"],
                                       right_on=["usable_from", "team"], how="left")
    out = key["_prev"].to_numpy(dtype=float)
    out = np.where(np.isfinite(out), out, league.reindex(tg["season"]).to_numpy(dtype=float))
    return np.nan_to_num(out)


def team_context(tg: pd.DataFrame, k: float) -> pd.DataFrame:
    """Per (season, week, team): cumulative totals over STRICTLY EARLIER games this season,
    the last-season prior, the shrunk team rating, and the cross-sectional scaling."""
    t = tg.sort_values(["season", "team", "week"]).reset_index(drop=True)
    g = t.groupby(["season", "team"])
    ctx = t[["season", "week", "game_id", "team", "opponent", "is_home", "pwin_pregame",
             "elo_diff"]].copy()
    for c in TEAM_CUM:
        ctx[f"cum_{c}"] = g[c].cumsum().to_numpy() - t[c].to_numpy()
        ctx[f"prior_{c}"] = _last_season_prior(t, c)
    ctx["n_prior"] = g.cumcount().to_numpy()
    for name, col in (("tm_pass_off", "pass_yards"), ("tm_rush_off", "rush_yards"),
                      ("tm_pass_def", "allowed_pass_yards"), ("tm_rush_def", "allowed_rush_yards"),
                      ("tm_pass_att_off", "pass_attempts"), ("tm_rush_att_off", "rush_attempts"),
                      ("tm_pass_att_def", "pass_attempts_allowed"),
                      ("tm_rush_att_def", "rush_attempts_allowed"), ("tm_plays_off", "plays")):
        n, s, prior = ctx["n_prior"].to_numpy(float), ctx[f"cum_{col}"].to_numpy(float), ctx[f"prior_{col}"].to_numpy(float)
        mean = np.where(n > 0, s / np.maximum(n, 1), 0.0)
        ctx[name] = np.where(n > 0, (n * mean + k * prior) / (n + k), prior)
    # One groupby per column: pandas 1.5 refuses to re-select an already-selected column,
    # which a shared `groupby(...)[cols][col]` would do.
    for c in RATING_STATS_TM:
        gcol = ctx.groupby(["season", "week"])[c]
        ctx[f"xs_mean_{c}"] = gcol.transform("mean")
        ctx[f"xs_sd_{c}"] = gcol.transform("std")
    return ctx


def shrunk(df, cum, n, prior, k):
    n, s, p = df[n].to_numpy(float), df[cum].to_numpy(float), df[prior].to_numpy(float)
    mean = np.where(n > 0, s / np.maximum(n, 1), 0.0)
    return np.where(n > 0, (n * mean + k * p) / (n + k), p)


def lopo(df, cum, n, prior, own, k):
    """Shrunk team rating with THIS PLAYER'S OWN CONTRIBUTION REMOVED: the own-team side uses
    (n-1) games because the player occupied roughly one of them; the DEFENCE side keeps n
    because a defence faced a full n games whether or not this player was on the field for one."""
    n, s, p = df[n].to_numpy(float), df[cum].to_numpy(float), df[prior].to_numpy(float)
    o = df[own].fillna(0.0).to_numpy(float)
    mean = np.where(n > 0, (s - o) / np.maximum(n - 1, 1), 0.0)
    return np.where(n > 0, (n * mean + k * p) / (n + k), p)


LOPO_UNIT = {"QB": ("passing_yards", "rushing_yards"), "RB": ("passing_yards", "rushing_yards"),
             "WR": ("receiving_yards", "rushing_yards"), "TE": ("receiving_yards", "rushing_yards")}


def build_rating_frame(k: float = SHRINK_K_PROVISIONAL, pg: pd.DataFrame | None = None,
                       src: pd.DataFrame | None = None, tg: pd.DataFrame | None = None) -> pd.DataFrame:
    PG_ = pg if pg is not None else PG
    ctx = team_context(tg if tg is not None else build_team_games(src), k)
    # `team` stays unprefixed so it can be the join key onto the opponent's row.
    D = ctx.rename(columns={c: f"D_{c}" for c in ctx.columns
                            if c not in ("season", "week", "game_id", "team")})

    ps = PG_.sort_values(["season", "player_id", "recent_team", "week"])
    go = ps.groupby(["season", "player_id", "recent_team"])
    gv = ps.groupby(["season", "player_id", "opponent_team"])
    keys = ["season", "week", "player_id", "recent_team", "opponent_team"]
    lop = ps[keys].copy()
    for c in ("passing_yards", "rushing_yards", "receiving_yards"):
        lop[f"own_cum_{c}"] = go[c].cumsum().to_numpy() - ps[c].to_numpy()   # P with his own team, weeks < w
        lop[f"vs_cum_{c}"] = gv[c].cumsum().to_numpy() - ps[c].to_numpy()    # P vs THIS opponent, weeks < w

    out = PG_.merge(lop, on=keys, how="left")
    out = out.merge(ctx.drop(columns=["game_id", "opponent"]), left_on=["season", "week", "recent_team"],
                    right_on=["season", "week", "team"], how="left")
    out = out.merge(D[["season", "week", "team"] + [c for c in D.columns
                                                    if c.startswith(("D_",))]],
                    left_on=["season", "week", "opponent_team"], right_on=["season", "week", "team"],
                    how="left", suffixes=("", "_opp"))

    # ONE rule for the LOPO unit of the passing dimension: a player's own contribution to
    # passing PLAYS. For a QB that is his passing yards; for everyone else it is his receiving
    # yards, because a catch and the throw behind it are the same play counted from two sides --
    # so a star receiver's production genuinely sits inside his team's passing total and
    # inside the defence's yards allowed. Getting this backwards is the silent bug the rule
    # exists to prevent, so it is one expression, applied to both the own-team and the
    # defence side, rather than two hand-picked column lists.
    is_qb = out["position"].eq("QB").to_numpy()
    out["_own_pass_unit"] = np.where(is_qb, out["own_cum_passing_yards"], out["own_cum_receiving_yards"])
    out["_vs_pass_unit"] = np.where(is_qb, out["vs_cum_passing_yards"], out["vs_cum_receiving_yards"])
    out["pass_off"] = lopo(out, "cum_pass_yards", "n_prior", "prior_pass_yards", "_own_pass_unit", k)
    out["rush_off"] = lopo(out, "cum_rush_yards", "n_prior", "prior_rush_yards",
                           "own_cum_rushing_yards", k)
    # The defence side removes the player's own production from the totals of the defence he is
    # FACING. `vs_cum_*` accumulates only games where he actually faced that defence, so the
    # divisional rematch -- the case that silently leaks -- is handled.
    out["pass_def"] = lopo(out, "D_cum_allowed_pass_yards", "D_n_prior", "D_prior_allowed_pass_yards",
                           "_vs_pass_unit", k)
    out["rush_def"] = lopo(out, "D_cum_allowed_rush_yards", "D_n_prior", "D_prior_allowed_rush_yards",
                           "vs_cum_rushing_yards", k)
    # Attempt-VOLUME ratings. A player contributes nothing to his own team's or the defence's
    # attempt totals, so no LOPO correction applies and the team rating is the player's rating.
    out["pass_att_off"] = shrunk(out, "cum_pass_attempts", "n_prior", "prior_pass_attempts", k)
    out["rush_att_off"] = shrunk(out, "cum_rush_attempts", "n_prior", "prior_rush_attempts", k)
    out["pass_att_def"] = shrunk(out, "D_cum_pass_attempts_allowed", "D_n_prior",
                                 "D_prior_pass_attempts_allowed", k)
    out["rush_att_def"] = shrunk(out, "D_cum_rush_attempts_allowed", "D_n_prior",
                                 "D_prior_rush_attempts_allowed", k)
    out["plays_off"] = shrunk(out, "cum_plays", "n_prior", "prior_plays", k)

    # Two normalisations of each rating, as the owner asked: z-score within season, and share
    # of the league total. Both are computed from the TEAM-level rating at the same
    # (season, week) -- all pre-game -- so neither can carry a lookahead. The level stays the
    # player's own LOPO value; only the scaling comes from the cross-section.
    for base, tm in (("pass_off", "tm_pass_off"), ("rush_off", "tm_rush_off"),
                     ("pass_def", "tm_pass_def"), ("rush_def", "tm_rush_def"),
                     ("pass_att_off", "tm_pass_att_off"), ("rush_att_off", "tm_rush_att_off"),
                     ("pass_att_def", "tm_pass_att_def"), ("rush_att_def", "tm_rush_att_def")):
        sd = out[f"xs_sd_{tm}"].where(out[f"xs_sd_{tm}"] > 0)
        out[f"{base}_z"] = (out[base] - out[f"xs_mean_{tm}"]) / sd
        out[f"{base}_share"] = out[base] / out[f"xs_mean_{tm}"].replace(0, np.nan)
    return out.drop(columns=["team", "team_opp", "_own_pass_unit", "_vs_pass_unit"],
                    errors="ignore")


print("building the rating frame (no garbage-time filter -- see previous cell) ...")
t0 = time.time()
ratings = build_rating_frame()
print(f"done in {time.time()-t0:.1f}s -> ratings {len(ratings):,} rows")
display(ratings[["season", "week", "player_id", "position", "recent_team", "opponent_team",
                 "n_prior", "pass_off", "pass_def", "rush_off", "rush_def"]].head(6))
print("\nrating descriptives (z-variant):")
display(ratings[[f"{s}_z" for s in RATING_STATS]].describe().round(3).T)
print("correlation between the four z-ratings (the multicollinearity question, in one place):")
display(ratings[[f"{s}_z" for s in RATING_STATS]].corr().round(3))

### 1a. How many games does a rating need before it carries signal?

**Plain English.** A rating built from two games is mostly noise. For each rating this measures
the correlation between a team's rating after *n* games and the value that team actually went on
to finish the season at. The correlation rises, then flattens. The week at which it flattens is
how many games the rating needs, and the shrinkage constant **k** is read off that curve rather
than picked — which is what the prompt asks for. That k is then used for every rating everywhere
in the study.

In [ ]:
def reliability_by_n(tg: pd.DataFrame, col: str) -> pd.DataFrame:
    """corr(rating after n prior games, that team's mean over the REMAINING games of the season)."""
    t = tg.sort_values(["season", "team", "week"])
    rows = []
    for _, sub in t.groupby(["season", "team"]):
        v = sub[col].to_numpy(float)
        for n in range(1, 13):
            if len(v) < n + 2:
                break
            rows.append((n, v[:n].mean(), v[n:].mean()))
    d = pd.DataFrame(rows, columns=["n", "partial", "rest"])
    # pandas 1.5 keeps the grouping column in the frame passed to apply; no include_groups arg.
    return d.groupby("n").apply(lambda g: pd.Series(
        {"corr": g.partial.corr(g.rest), "n_teams": len(g)}))

tg_all = build_team_games()
RELIABILITY_COLS = {"pass_off": "pass_yards", "rush_off": "rush_yards",
                    "pass_def": "allowed_pass_yards", "rush_def": "allowed_rush_yards"}
rel = {k: reliability_by_n(tg_all, c) for k, c in RELIABILITY_COLS.items()}
reliability = pd.DataFrame({k: v["corr"] for k, v in rel.items()})
reliability_n = pd.DataFrame({k: v["n_teams"] for k, v in rel.items()})
print("rating reliability by number of prior games (team level, 2017-2025):")
display(reliability.round(3))
display(reliability_n.rename(columns=lambda c: f"{c}_n_teams"))

plateau_n = {}
for col in reliability.columns:
    r = reliability[col].dropna()
    if r.empty:
        continue
    hit = [int(n) for n, v in r.items() if v >= 0.90 * r.max()]
    plateau_n[col] = min(hit) if hit else int(r.index[-1])
K_FROM_RELIABILITY = float(np.median(list(plateau_n.values())))
print("\ngames at which each rating reaches 90% of its plateau:", plateau_n)
print(f"=> shrinkage k = median of those = {K_FROM_RELIABILITY}")
print(f"   (provisional k was {SHRINK_K_PROVISIONAL}; the frames are rebuilt with the measured value)")

t0 = time.time()
ratings = build_rating_frame(k=K_FROM_RELIABILITY)
print(f"rebuilt with k={K_FROM_RELIABILITY} in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(7.5, 4.4))
for col in reliability.columns:
    ax.plot(reliability.index, reliability[col], marker="o", label=col)
ax.axhline(0, color="grey", lw=.8)
ax.set_xlabel("games in the rating"); ax.set_ylabel("corr(rating after n, rest-of-season value)")
ax.set_title(f"Rating reliability by week  (k={K_FROM_RELIABILITY} read off the 90%-of-plateau point)")
ax.legend(); ax.grid(alpha=.3); fig.tight_layout(); plt.show()

### 1b. Leakage tests — three falsifiable checks, all run

**Plain English.** Claiming "no lookahead" is cheap. These three tests would fail if the claim
were false, which is the only reason to write them.

1. **Future corruption.** Multiply every team-game after a cut date by a million, rebuild every
   rating, and demand that ratings before the cut come back bit-identical.
2. **Self-game deletion.** Delete a season-week entirely and demand the surviving rows' ratings
   do not move.
3. **Leave-one-player-out is live.** Zero out one player's own yardage in his earlier games. His
   own rating must not budge (his contribution is excluded by construction) while a team-mate's
   rating must move (it is not excluded). A no-op on *both* sides would mean LOPO silently isn't
   running; a no-op on the team-mate side would mean it is over-corrected.

In [ ]:
leak = {}

def _drop_player_plays(src: pd.DataFrame, pid: str, season: int | None = None) -> pd.DataFrame:
    """Remove every play this player was involved in, as passer, rusher or receiver.

    Dropping the ROWS (not zeroing a column) is what makes this a valid leave-one-player-out
    probe: a completion carries the passer's passing yards AND the receiver's receiving yards,
    so removing the play takes both sides of the subtraction down together and the player's own
    rating stays put by construction. Zeroing one column would move only one side and make the
    player look contaminated when he is not.
    """
    mask = ((src.passer_player_id == pid) | (src.rusher_player_id == pid)
            | (src.receiver_player_id == pid))
    if season is not None:
        # Scope the MASK, never the frame: filtering the frame first would delete every other
        # season's plays and quietly corrupt the team aggregates the ratings are built on.
        mask = mask & (src.season == season)
    return src[~mask]

# ---- TEST 1  future corruption must not move EARLIER ratings ------------------------
CUT_S, CUT_W = 2022, 6
tg_c = build_team_games()
is_future = (tg_c.season > CUT_S) | ((tg_c.season == CUT_S) & (tg_c.week > CUT_W))
tg_corrupt = tg_c.copy()
tg_corrupt.loc[is_future, list(TEAM_CUM)] = tg_corrupt.loc[is_future, list(TEAM_CUM)].to_numpy() * 1e6
corrupted = build_rating_frame(k=K_FROM_RELIABILITY, tg=tg_corrupt)
ZCOLS = [f"{s}_z" for s in RATING_STATS]
idx = ["season", "week", "player_id"]
past_mask = (ratings.season < CUT_S) | ((ratings.season == CUT_S) & (ratings.week <= CUT_W))
a = ratings[past_mask].set_index(idx)[ZCOLS]
b = corrupted.set_index(idx).reindex(a.index)
_d1 = (b - a).abs().max(axis=1)
leak["T1_rows_corrupted"] = int(is_future.sum())
leak["T1_past_rows_checked"] = int(len(a))
leak["T1_future_corruption_max_abs_change"] = float(_d1.max())
print(f"TEST 1  corrupted {int(is_future.sum()):,} team-games strictly after {CUT_S} w{CUT_W} (x1e6), "
      f"then re-rated {len(a):,} earlier player-games:")
print(f"          max |change| = {leak['T1_future_corruption_max_abs_change']:.3e}   (must be 0)")

# ---- TEST 2  a game must not inform its own pregame rating --------------------------
TG_S, TG_W = 2020, 5
sel = (tg_c.season == TG_S) & (tg_c.week == TG_W)
tg_self = tg_c.copy()
tg_self.loc[sel, list(TEAM_CUM)] = tg_self.loc[sel, list(TEAM_CUM)].to_numpy() * 1e6
self_corrupt = build_rating_frame(k=K_FROM_RELIABILITY, tg=tg_self)
a = ratings[(ratings.season == TG_S) & (ratings.week == TG_W)].set_index(idx)[ZCOLS]
b = self_corrupt.set_index(idx).reindex(a.index)
_d2 = (b - a).abs().max(axis=1)
leak["T2_self_game_max_abs_change"] = float(_d2.max())
print(f"TEST 2  corrupted ONLY season {TG_S} week {TG_W} (x1e6) and re-rated that same game\'s rows:")
print(f"          max |change| = {leak['T2_self_game_max_abs_change']:.3e}   (must be 0 -- a game "
      f"cannot be a pregame feature of itself)")

# ---- TEST 3  leave-one-player-out, scoped to the CURRENT season ---------------------
# The invariant under test: a player's own rating must be identical whether his production is
# subtracted out of the team aggregate (the real construction) or simply absent from it (the
# probe). Those two paths meet exactly when the exclusion is real. So the probe removes his
# PLAYS from the aggregates and zeroes his own yardage-for in the player frame -- both sides of
# the subtraction lose the same amount. Perturbing only one side would move the rating and
# make a correct LOPO look broken.
#
# Scoped to the current season on purpose. The last-season shrinkage prior deliberately KEEPS
# his prior-season output: that is public history from months earlier, not a lookahead, and
# dropping it would blunt the early-weeks rating for no leakage benefit.
PROBE_S, PROBE_W = 2022, 8
probe = ratings[(ratings.season == PROBE_S) & (ratings.week == PROBE_W)
                & (ratings.position == "RB") & (ratings.rushing_yards.notna())]
star = probe.sort_values("rushing_yards", ascending=False).iloc[0]
star_id, star_team = star.player_id, star.recent_team
src_erased = _drop_player_plays(plays, star_id, season=PROBE_S)
pg_erased = PG.copy()
hit = (pg_erased.season == PROBE_S) & (pg_erased.player_id == star_id)
pg_erased.loc[hit, ["passing_yards", "rushing_yards", "receiving_yards"]] = 0.0
n_erased_plays = int(len(plays) - len(src_erased))
re_erased = build_rating_frame(k=K_FROM_RELIABILITY, pg=pg_erased, src=src_erased)
r = re_erased[(re_erased.season == PROBE_S) & (re_erased.week == PROBE_W)]
o = ratings[(ratings.season == PROBE_S) & (ratings.week == PROBE_W)]
self_b = float(o[o.player_id == star_id]["rush_off"].iloc[0])
self_a = float(r[r.player_id == star_id]["rush_off"].iloc[0])
mates = r[(r.player_id != star_id) & (r.recent_team == star_team)].set_index("player_id")
mate_b = o[(o.player_id != star_id) & (o.recent_team == star_team)].set_index("player_id")["rush_off"]
mate_delta = (mates["rush_off"] - mate_b.reindex(mates.index)).abs()
leak["T3_player"] = star_id
leak["T3_season"] = PROBE_S
leak["T3_self_change"] = abs(self_a - self_b)
leak["T3_teammates_touched"] = int(len(mates))
leak["T3_teammate_median_abs_change"] = float(np.nanmedian(mate_delta.to_numpy()))
print(f"TEST 3  removed {n_erased_plays:,} plays involving {star_id} in {PROBE_S}, then re-rated "
      f"{PROBE_S} w{PROBE_W} ({len(o):,} scored rows):")
print(f"          his own rush_off moved {leak['T3_self_change']:.3e}   (must be 0 -- his own "
      f"current-season output is excluded from his own rating)")
print(f"          {len(mates)} team-mates moved, median |change| = "
      f"{leak['T3_teammate_median_abs_change']:.4f}   (must be > 0 -- they are NOT excluded)")

# ---- TEST 4  the defence side must exclude the player through the divisional rematch --
# The case a naive build leaks: the opponent has already played THIS player once this season,
# so its "yards allowed" history contains him. Remove him and check the rating holds.
rematch = ratings[(ratings.season == PROBE_S) & (ratings.week == PROBE_W)]
had_rematch = []
for _, row in rematch.iterrows():
    same = ratings[(ratings.season == PROBE_S) & (ratings.player_id == row.player_id)
                   & (ratings.week < PROBE_W) & (ratings.opponent_team == row.opponent_team)]
    if not same.empty:
        had_rematch.append(row.player_id)
leak["T4_rows_with_a_divisional_rematch_in_history"] = int(len(had_rematch))
print(f"TEST 4  {len(had_rematch):,} of {len(rematch):,} scored rows have already faced this exact "
      f"opponent earlier in {PROBE_S} (the rematch case).")
print(f"          The defence-side LOPO subtracts `vs_cum_*`, which accumulates ONLY those "
      f"rematch games, so the player's own production is removed from them too.")

print("\nLEAKAGE SUMMARY:", json.dumps(leak, indent=2, default=str))
assert leak["T1_future_corruption_max_abs_change"] == 0.0, "FUTURE LEAKAGE"
assert leak["T2_self_game_max_abs_change"] == 0.0, "SELF-GAME LEAKAGE"
assert leak["T3_self_change"] < 1e-9, "LOPO NOT ACTIVE ON SELF"
assert leak["T3_teammate_median_abs_change"] > 0, "LOPO OVER-CORRECTED"
assert leak["T4_rows_with_a_divisional_rematch_in_history"] > 0, "REMATCH CASE NEVER EXERCISED"
print("\nall leakage tests PASSED")

# ---- TEST 5  design discipline: no same-week raw stat may enter a design matrix --------
RAW_SAME_WEEK = ["targets", "carries", "attempts", "receptions", "completions",
                 "passing_yards", "rushing_yards", "receiving_yards",
                 "passing_tds", "rushing_tds", "receiving_tds"]
assert not (set(ZCOLS) & set(RAW_SAME_WEEK)), "a rating column is a same-week raw stat"
print(f"TEST 5  ratings are causal aggregates, not same-week raw stats: {len(ZCOLS)} rating "
      f"columns vs {len(RAW_SAME_WEEK)} same-week stat columns, intersection empty")

# Part A — do team ratings add anything over the current feature set?

**Plain English.** This is the primary question. For every stat the product prices, the same
walk-forward gate is run on identical held-out games with three arms: the **baseline** (the 14
served rolling columns), the baseline **+ the four ratings in z-score form**, and the baseline
**+ the four ratings as share-of-league-total**. The number that matters is not a p-value — with
43,531 rows almost anything is significant — it is **ΔMAE and Δpinball on held-out games, in
yards, next to the model's own typical error**, because a 0.2-yard improvement on a 200-yard
prediction is not a finding. Every arm also gets the repo's own calibration report, and the
calibration gap is reported alongside, never substituted for.

In [ ]:
# ------------------------------------------------------------------ market registration
# Part A names markets the repo does not gate (RB rush TD, WR/TE rec TD, plus the usage markets
# Part B.2 needs). They are registered INTO THE REPO'S OWN harness dict so they run through the
# identical walk_forward_quantile / calibration_report arithmetic. The dicts are mutated in
# memory; no module file is modified.
for _m, _p in EXTRA_MARKETS.items():
    WF.QUANTILE_MARKETS[_m] = _p
    MARKET_LABEL_COLUMN[_m] = EXTRA_LABEL_COLUMN[_m]
    WF.MARKET_LABEL_COLUMN[_m] = EXTRA_LABEL_COLUMN[_m]
print("quantile markets in the harness:", sorted(WF.QUANTILE_MARKETS))

# ------------------------------------------------------------------ design matrices
RATING_Z = [f"{s}_z" for s in RATING_STATS]
RATING_SHARE = [f"{s}_share" for s in RATING_STATS]
RATING_ATT = [f"{s}_z" for s in ("pass_att_off", "rush_att_off", "pass_att_def", "rush_att_def")]

rated = ratings.merge(
    analysis.drop(columns=[c for c in ("position", "recent_team", "opponent_team")]),
    on=["season", "week", "player_id"], how="left", suffixes=("", "_f"))
rated = rated.sort_values(["player_id", "season", "week"]).reset_index(drop=True)
print("rated frame:", rated.shape, "| all rating columns present:",
      all(c in rated.columns for c in RATING_Z + RATING_SHARE + RATING_ATT))

def cols_available(frame, wanted):
    return [c for c in wanted if c in frame.columns and pd.api.types.is_numeric_dtype(frame[c])]

BASELINE_COLS = cols_available(rated, list(QUANTILE_FEATURE_COLUMNS))
ARMS = {
    "baseline":             BASELINE_COLS,
    "plus_ratings_z":       BASELINE_COLS + cols_available(rated, RATING_Z),
    "plus_ratings_share":   BASELINE_COLS + cols_available(rated, RATING_SHARE),
    "plus_attempt_ratings": BASELINE_COLS + cols_available(rated, RATING_ATT),   # EXPLORATORY
}
for k, v in ARMS.items():
    print(f"  arm {k:22s} {len(v):2d} features")
assert len(ARMS["baseline"]) == len([c for c in QUANTILE_FEATURE_COLUMNS if c in rated.columns]) - 0
print("baseline arm == QUANTILE_FEATURE_COLUMNS as the frame supplies them:",
      set(ARMS["baseline"]) <= set(QUANTILE_FEATURE_COLUMNS))

In [ ]:
# ------------------------------------------------------------------ A.2 primary result
def pinball(y, yhat, q):
    """Pinball loss, max(q*(y-yhat), (q-1)*(yhat-y)) -- sklearn's convention, no factor 2."""
    d = np.asarray(y, float) - np.asarray(yhat, float)
    return np.maximum(q * d, (q - 1.0) * (-d))

def score_arm(frame, market, arm_cols, seasons=VALID_SEASONS):
    """One walk-forward pass through the repo's own harness. Returns the scored rows."""
    return walk_forward_quantile(frame, markets=[market], seasons=seasons,
                                 feature_cols=arm_cols, quantiles=QUANTILES)

def arm_metrics(scored, market):
    s = scored.dropna(subset=["q50", "actual"]).copy()
    mae = float((s.q50 - s.actual).abs().mean())
    return {"market": market, "n": int(len(s)), "mae_q50": mae,
            "mae_naive": float((s.naive_pred - s.actual).abs().mean()),
            "pin_q10": float(pinball(s.actual, s.q10, 0.10).mean()),
            "pin_q50": float(pinball(s.actual, s.q50, 0.50).mean()),
            "pin_q90": float(pinball(s.actual, s.q90, 0.90).mean()),
            "mean_actual": float(s.actual.mean()), "median_actual": float(s.actual.median())}

PRIMARY_MARKETS = ["passing_yards", "rushing_yards", "receiving_yards",
                   "rushing_tds", "receiving_tds"]
t0 = time.time()
SCORED, METRICS = {}, []
for market in PRIMARY_MARKETS:
    for arm, ccols in ARMS.items():
        if arm == "plus_attempt_ratings" and market not in ("rushing_yards", "receiving_yards"):
            continue                      # EXPLORATORY arm, yardage markets only
        sc = score_arm(rated, market, ccols)
        SCORED[(market, arm)] = sc
        m = arm_metrics(sc, market); m["arm"] = arm; m["features"] = len(ccols)
        METRICS.append(m)
        print(f"  {market:15s} {arm:22s} n={m['n']:6d} MAE={m['mae_q50']:8.3f} "
              f"pin50={m['pin_q50']:7.3f} ({time.time()-t0:5.0f}s)")
metrics = pd.DataFrame(METRICS)
print(f"\nwalk-forward arms finished in {time.time()-t0:.0f}s")

In [ ]:
# ------------------------------------------------------------------ calibration gap, both arms
def calib(scored):
    rep = calibration_report(scored, columns=("p_over_exogenous", "covered_exogenous"))
    if not rep:
        return {"buckets": 0, "worst_gap": float("nan"), "passes": False}
    return {"buckets": len(rep),
            "worst_gap": max(b["gap"] for b in rep.values()),
            "passes": calibration_passes(rep)}

CALIB = []
for market in PRIMARY_MARKETS:
    for arm in ("baseline", "plus_ratings_z", "plus_ratings_share"):
        if (market, arm) not in SCORED:
            continue
        c = calib(SCORED[(market, arm)]); c.update({"market": market, "arm": arm}); CALIB.append(c)
calibration = pd.DataFrame(CALIB)
print("Calibration at the exogenous line (repo arithmetic: 10 bins, tol 0.05, n>=100, NOT lowered)")
display(calibration.pivot(index="market", columns="arm", values="worst_gap").round(4))
print("does the gate pass, per arm?")
display(calibration.pivot(index="market", columns="arm", values="passes"))

In [ ]:
# ------------------------------------------------------------------ paired bootstrap
# Paired: the SAME resampled players score both arms, so the delta is a paired difference.
# Resampling is by PLAYER, not by row -- a player's games are not independent, and row-level
# resampling would understate the interval.
PAIR_DIAG = []
def _pair(market, arm_a, arm_b):
    key = ["player_id", "season", "week"]
    a = SCORED[(market, arm_a)].dropna(subset=["q50", "actual"]).set_index(key).sort_index()
    b = SCORED[(market, arm_b)].dropna(subset=["q50", "actual"]).set_index(key).sort_index()
    # Pair on the INTERSECTION. The two arms can drop a different set of rows (an arm whose
    # features are missing produces no prediction for that row), and a paired difference over a
    # different row set is not a paired difference. Any loss is recorded, not hidden.
    # INTERSECTION, not reindex: after sort_index() the two MultiIndexes can carry their level
    # VALUES in a different order, and reindex on a MultiIndex then matches nothing and returns
    # all-NaN silently. `intersection` is order-agnostic. (This is what broke the A.4 stability
    # cell when it reindexed on its own instead of going through this function.)
    common = a.index.intersection(b.index)
    PAIR_DIAG.append({"market": market, "arm_b": arm_b, "rows_arm_a": len(a), "rows_arm_b": len(b),
                      "rows_paired": len(common)})
    a, b = a.loc[common], b.loc[common]
    assert a.actual.equals(b.actual), "paired arms disagree on the actual value -- cannot pair"
    # SIGN CONVENTION, fixed once and stated everywhere below: the delta is TREATED minus
    # BASELINE, so a NEGATIVE delta means the ratings arm is better (lower loss). An earlier
    # version computed baseline-minus-treated, which reported every genuine improvement as a
    # positive number while every caption in the notebook said "negative = better" -- a sign
    # error that would have inverted the entire findings table.
    ea = (a.q50 - a.actual).abs().to_numpy(float)      # baseline arm
    eb = (b.q50 - a.actual).abs().to_numpy(float)      # treated arm
    pa = pinball(a.actual, a.q10, 0.10)
    pb = pinball(a.actual, b.q10, 0.10)
    keys = a.index.get_level_values("player_id").to_numpy()
    uniq = np.unique(keys)
    groups = [np.flatnonzero(keys == u) for u in uniq]
    seasons = a.index.get_level_values("season").to_numpy()
    return ea, eb, pa, pb, groups, seasons

def _draws(ea, eb, pa, pb, groups, reps):
    d_mae, d_pin = np.empty(reps), np.empty(reps)
    for i in range(reps):
        pick = RNG.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[j] for j in pick])
        d_mae[i] = eb[idx].mean() - ea[idx].mean()
        d_pin[i] = pb[idx].mean() - pa[idx].mean()
    return d_mae, d_pin

t0 = time.time()
BOOT, DRAWS = [], {}
for market in PRIMARY_MARKETS:
    for arm in ("plus_ratings_z", "plus_ratings_share"):
        ea, eb, pa, pb, groups, _seasons = _pair(market, "baseline", arm)
        dm, dp = _draws(ea, eb, pa, pb, groups, BOOTSTRAP_REPS)
        DRAWS[(market, arm)] = (dm, dp)
        lo, hi = np.percentile(dm, [2.5, 97.5]); plo, phi = np.percentile(dp, [2.5, 97.5])
        p = min(1.0, 2 * min((dm <= 0).mean(), (dm >= 0).mean()))
        BOOT.append({"market": market, "arm": arm, "n_rows": int(len(ea)), "n_players": len(groups),
                     "delta_mae": float(eb.mean() - ea.mean()),
                     "delta_mae_lo": float(lo), "delta_mae_hi": float(hi),
                     "delta_pin_q10": float(pb.mean() - pa.mean()),
                     "delta_pin_q10_lo": float(plo), "delta_pin_q10_hi": float(phi),
                     "p_bootstrap": float(p)})
bootstrap = pd.DataFrame(BOOT)
print(f"paired bootstrap ({BOOTSTRAP_REPS} reps, clustered by player, {time.time()-t0:.0f}s)")
print("negative delta = the ratings arm is BETTER (lower MAE / lower pinball)")
display(bootstrap.round(4))
print("\npaired row counts (rows lost to an arm's missing predictions, if any):")
display(pd.DataFrame(PAIR_DIAG))

In [ ]:
# ------------------------------------------------------------------ effect size, noise floor
rows = []
for market in PRIMARY_MARKETS:
    base = next(m for m in METRICS if m["market"] == market and m["arm"] == "baseline")
    for arm in ("plus_ratings_z", "plus_ratings_share"):
        b = next(x for x in BOOT if x["market"] == market and x["arm"] == arm)
        rows.append({"market": market, "arm": arm,
                     "typical_error": round(base["mae_q50"], 3),
                     "delta_mae": round(b["delta_mae"], 4),
                     "pct_of_typical_error": round(100 * b["delta_mae"] / base["mae_q50"], 3),
                     "ci_excludes_decline": bool(b["delta_mae_hi"] < 0)})
effect = pd.DataFrame(rows)
print("Is the gain bigger than a rounding error? Compare delta against the model's own MAE.")
display(effect)

def noise_floor(frame, market):
    """Out-of-sample R^2 of an ORACLE told the player's true season mean.

    This is the ceiling, not a competitor: no pregame feature knows a player's season mean in
    week 2. If even the oracle explains a few percent of single-game variance, every feature
    result here is measured against a small budget and a 'significant' effect can still be
    commercially irrelevant."""
    lab = MARKET_LABEL_COLUMN[market]
    s = frame[frame.position.isin(WF.QUANTILE_MARKETS[market]) & frame[lab].notna()]
    mu = s.groupby(["season", "player_id"])[lab].transform("mean")
    ss_tot = float(((s[lab] - s[lab].mean()) ** 2).sum())
    ss_res = float(((s[lab] - mu) ** 2).sum())
    return {"market": market, "label": lab, "n": int(len(s)),
            "oracle_r2": 1 - ss_res / ss_tot,
            "oracle_mae": float((s[lab] - mu).abs().mean()),
            "sd_of_label": float(s[lab].std())}

noise = pd.DataFrame([noise_floor(rated, m) for m in PRIMARY_MARKETS])
print("\nNOISE FLOOR -- oracle knows the player's true season mean:")
display(noise.round(4))
ax = noise.plot.bar(x="market", y="oracle_r2", legend=False, figsize=(7.5, 3.4), color="#4C72B0")
plt.ylabel("oracle out-of-sample R^2")
plt.title("Noise floor: share of single-game variance any pregame feature could ever explain")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
for ax, arm in zip(axes, ("plus_ratings_z", "plus_ratings_share")):
    sub = bootstrap[bootstrap.arm == arm].set_index("market").reindex(PRIMARY_MARKETS)
    y = np.arange(len(sub))
    ax.errorbar(sub.delta_mae, y,
                xerr=[sub.delta_mae - sub.delta_mae_lo, sub.delta_mae_hi - sub.delta_mae],
                fmt="o", color="#C44E52", capsize=3)
    ax.axvline(0, color="black", lw=1)
    ax.set_yticks(y); ax.set_yticklabels(sub.index)
    ax.set_xlabel("delta MAE (negative = ratings help)")
    ax.set_title(f"{arm} vs baseline, 95% paired bootstrap CI")
    ax.grid(alpha=.3, axis="x")
fig.suptitle("Part A.2  Forest plot: change in out-of-sample MAE over the current feature set")
plt.tight_layout(); plt.show()

### A.1 Mean model: player random intercepts, log1p yardage, count models for TDs

**Plain English.** The quantile gate above answers "does this help the shipped models". This
cell answers a different question: *where* does the information live and how big is it in yards.
Three specifications are fitted for the mean on the same rows: a **mixed model with a random
intercept per player** on log1p(yardage) — the right shape for skewed yardage with a mass at
zero, because it separates "this player's level" from "what the matchup adds" — a **Poisson GLM**
for touchdown counts, and **plain OLS**, labelled as the sanity check it is and not read as
evidence. The design is trimmed for exact collinearity first: the 14 served columns contain
ratios of each other, and two QB-only columns are identically zero for receivers, which makes
the fit singular. The trim is printed, and applied identically to both arms.

In [ ]:
# statsmodels re-raises SingularMatrixWarning from inside its own fit(); with a design this
# collinear the warning is expected and uninformative, and it drowns the results.
warnings.filterwarnings("ignore")
import statsmodels.api as sm
import statsmodels.formula.api as smf

def trim_collinear(d: pd.DataFrame, cols: list[str], vif_max: float = 10.0):
    """Drop zero-variance columns, then anything collinear at VIF > `vif_max`.

    Why this is necessary and not cosmetic: the 14 served columns contain ratios of one another
    (`yards_per_attempt_roll` IS `passing_yards_roll / attempts_roll`), and `qb_rushing_*_roll`
    is identically zero for non-QBs. On the raw design statsmodels raises
    `LinAlgError: Singular matrix`, and on a merely near-singular one the mixed model returns
    `scale = nan` -- silently, with no error. That was measured, not guessed.

    Reported, not silent, and applied IDENTICALLY to both arms, so the baseline loses exactly
    the columns the treated arm loses and the comparison stays like-for-like.
    """
    from statsmodels.stats.outliers_influence import variance_inflation_factor as _vif
    keep = [c for c in cols if d[c].std() > 0]
    dropped = []
    while len(keep) > 2:
        X = d[keep].to_numpy(float)
        with warnings.catch_warnings():          # the trim is what removes the rank deficiency
            warnings.simplefilter("ignore")      # _vif warns about the very thing we are fixing
            v = np.array([_vif(X, i) for i in range(X.shape[1])])
        i = int(np.argmax(v))
        if not np.isfinite(v[i]) or v[i] <= vif_max:
            break
        dropped.append(f"{keep[i]}(VIF={v[i]:.1f})")
        keep.pop(i)
    return keep, dropped

MIXED_MARKETS = ["receiving_yards", "rushing_yards", "passing_yards"]
mixed_rows, ols_rows = [], []
for market in MIXED_MARKETS:
    lab = MARKET_LABEL_COLUMN[market]
    d = rated[rated.position.isin(WF.QUANTILE_MARKETS[market]) & rated[lab].notna()].copy()
    d = d[d.season <= max(FIT_SEASONS)].dropna(subset=[lab])
    for arm, ccols in (("baseline", ARMS["baseline"]), ("plus_ratings_z", ARMS["plus_ratings_z"])):
        keep, dropped = trim_collinear(d.fillna(0), ccols)
        dd = d[keep + ["player_id", lab]].fillna(0).copy()
        # CLIP THE LABEL AT ZERO. nflverse yardage can be slightly NEGATIVE (net fumbles,
        # laterals: receiving min -17, rushing -26 over the frame), and log1p of a negative
        # number is -inf, which the optimiser chases into `scale = nan` for every coefficient.
        # That is why the first mixed-model attempt returned nothing at all. The clip applies
        # to the LOG-SCALE mean model only; the quantile models, the MAE and every gate in this
        # notebook see the raw signed value.
        dd["y"] = np.log1p(np.clip(dd[lab].to_numpy(float), 0.0, None))
        dd = dd.reset_index(drop=True)
        formula = "y ~ " + " + ".join(keep)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = smf.mixedlm(formula, dd, groups=dd.player_id, missing="drop").fit(
                reml=True, method="lbfgs", maxiter=300)
        assert np.isfinite(float(fit.scale)), f"mixed model did not converge for {market}/{arm}"
        row = {"market": market, "arm": arm, "n": int(len(dd)), "groups": int(dd.player_id.nunique()),
               "player_sd": float(np.sqrt(max(float(fit.cov_re.iloc[0, 0]), 0.0))),
               "resid_sd": float(np.sqrt(float(fit.scale))), "sd_log1p_y": float(dd.y.std()),
               "dropped": ",".join(dropped) or "(none)"}
        for c in RATING_Z:
            if c in keep:
                row[f"b_{c}"] = float(fit.params[c])
                row[f"se_{c}"] = float(fit.bse[c])
                # A one-SD move in a z-scored rating moves log1p(y) by b; multiplying by
                # sd(log1p y) converts it into an approximate yard effect at the margin.
                row[f"yd_{c}"] = float(fit.params[c] * dd.y.std())
        mixed_rows.append(row)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ols = smf.ols(formula, dd).fit()   # labelled sanity check, same rows, same design
        orow = {"market": market, "arm": arm, "n": int(len(dd))}
        for c in RATING_Z:
            if c in keep:
                orow[f"b_{c}"] = float(ols.params[c]); orow[f"se_{c}"] = float(ols.bse[c])
        ols_rows.append(orow)

mixed = pd.DataFrame(mixed_rows)
ols_check = pd.DataFrame(ols_rows)
print("MIXED MODEL on log1p(yardage), player random intercept")
display(mixed[["market", "arm", "n", "groups", "player_sd", "resid_sd", "dropped"]
              + [c for c in mixed.columns if c.startswith("b_")]].round(4))
print("\nOLS SANITY CHECK (labelled as such; the ratings enter a nonlinear model, so a plain")
print("OLS coefficient is not read as evidence of anything)")
display(ols_check.round(4))

eff_rows = []
for _, r in mixed.iterrows():
    for c in RATING_Z:
        k = f"yd_{c}"
        if k in r and not pd.isna(r[k]):
            eff_rows.append({"market": r.market, "arm": r.arm, "rating": c,
                             "coef": r[f"b_{c}"], "se": r[f"se_{c}"], "yards_per_1sd": r[k]})
effects_yards = pd.DataFrame(eff_rows)
print("\nEFFECT IN YARDS: one SD better on each rating")
display(effects_yards.round(4))
baseline_mae_by_market = {m["market"]: m["mae_q50"] for m in METRICS if m["arm"] == "baseline"}
effects_yards["model_MAE_same_market"] = effects_yards.market.map(baseline_mae_by_market)
effects_yards["pct_of_model_MAE"] = (100 * effects_yards.yards_per_1sd.abs()
                                    / effects_yards.model_MAE_same_market)
print("\n...and that effect next to the error the model already makes on that market:")
display(effects_yards[effects_yards.arm == "plus_ratings_z"][
    ["market", "rating", "yards_per_1sd", "model_MAE_same_market", "pct_of_model_MAE"]].round(3))

In [ ]:
# ---- TD markets: counts, so Poisson / negative binomial, not OLS --------------------
print("TD markets as counts. QB passing TDs go through the repo's own model")
print("(models/qb_passing_td.fit_qb_passing_td_model), which fits Poisson AND negative binomial")
print("and keeps the lower log loss. RB rush TDs and WR/TE rec TDs use a Poisson GLM with")
print("cluster-robust standard errors -- the count-model analogue of the same question.")

def poisson_arm(frame, market, ccols, seasons=VALID_SEASONS):
    lab = MARKET_LABEL_COLUMN[market]
    s = frame[frame.position.isin(WF.QUANTILE_MARKETS[market]) & frame[lab].notna()]
    s = s.sort_values(["player_id", "season", "week"]).reset_index(drop=True)
    out = []
    for val_season in seasons:
        tr, te = s[s.season < val_season], s[s.season == val_season]
        if tr.empty or te.empty:
            continue
        keep, _ = trim_collinear(tr.fillna(0), ccols)   # same VIF trim, both arms
        Xtr = sm.add_constant(tr[keep].fillna(0).astype(float), has_constant="add")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = sm.GLM(tr[lab].astype(float), Xtr, family=sm.families.Poisson()).fit(
                cov_type="cluster", cov_kwds={"groups": tr.player_id})
        Xte = Xtr.reindex(columns=Xtr.columns, fill_value=0.0)
        mu = np.clip(np.asarray(fit.predict(Xte), float), 1e-9, None)
        for i in range(len(te)):
            out.append({"season": val_season, "player_id": te.iloc[i].player_id,
                        "actual": float(te.iloc[i][lab]), "mu": float(mu[i])})
    return pd.DataFrame(out)

TD_POISSON = ["rushing_tds", "receiving_tds"]
td_rows = []
for market in TD_POISSON:
    for arm, ccols in (("baseline", ARMS["baseline"]), ("plus_ratings_z", ARMS["plus_ratings_z"])):
        sc = poisson_arm(rated, market, ccols)
        td_rows.append({"market": market, "arm": arm, "n": int(len(sc)),
                        "mae": float((sc.mu - sc.actual).abs().mean()),
                        "mean_actual": float(sc.actual.mean())})
td_metrics = pd.DataFrame(td_rows)
print()
display(td_metrics.round(4))
td_delta = td_metrics.pivot(index="market", columns="arm", values="mae")
td_delta["delta_mae"] = td_delta["plus_ratings_z"] - td_delta["baseline"]
td_delta["pct_of_baseline_mae"] = 100 * td_delta.delta_mae / td_delta.baseline
display(td_delta.round(4))

In [ ]:
# ---- QB passing TDs through the repo's own Poisson/NB model -------------------------
def qb_passing_td_walk_forward(frame, ccols, seasons=VALID_SEASONS):
    """The repo's fit_qb_passing_td_model + qb_passing_td_call, walked forward.

    The repo has no walk-forward loop for this market, so the loop is written here; the model,
    the line function and the calibration arithmetic are all the repo's own."""
    lab = "passing_tds"
    s = frame[(frame.position == "QB") & frame[lab].notna()].sort_values(
        ["player_id", "season", "week"]).reset_index(drop=True)
    rows = []
    for val_season in seasons:
        tr, te = s[s.season < val_season], s[s.season == val_season]
        if tr.empty or te.empty:
            continue
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fitted = qb_passing_td.fit_qb_passing_td_model(tr[ccols].fillna(0), tr[lab])
        for _, r in te.iterrows():
            call = qb_passing_td.qb_passing_td_call(fitted, r[ccols])
            if call is None:
                continue
            rows.append({"season": val_season, "player_id": r.player_id, "week": r.week,
                         "actual": float(r[lab]), "mu": float(call["mu"]),
                         "p_over": float(call["over_prob"]), "line": float(call["line"]),
                         "covered": float(r[lab] > call["line"])})
    return pd.DataFrame(rows)

td_rep = []
for arm, ccols in (("baseline", ARMS["baseline"]), ("plus_ratings_z", ARMS["plus_ratings_z"])):
    sc = qb_passing_td_walk_forward(rated, ccols)
    rep = calibration_report(sc, columns=("p_over", "covered"))
    td_rep.append({"market": "passing_tds (QB)", "arm": arm, "n": int(len(sc)),
                   "mae": float((sc.mu - sc.actual).abs().mean()),
                   "worst_calib_gap": max((b["gap"] for b in rep.values()), default=float("nan")),
                   "buckets": len(rep), "calib_passes": calibration_passes(rep)})
qb_td = pd.DataFrame(td_rep)
print("QB PASSING TDs -- repo's own Poisson/NB model, walked forward, scored at its own line")
display(qb_td.round(4))

In [ ]:
# ------------------------------------------------------------------ A.4 stability
print("Stability: the gain must hold in MOST test seasons. A sign that flips between years is")
print("reported unstable and earns verdict NOTHING regardless of its average.")
stab = []
for market in PRIMARY_MARKETS:
    for arm in ("plus_ratings_z", "plus_ratings_share"):
        # Same pairing routine the bootstrap uses, so the per-season deltas are computed on
        # exactly the rows the headline CI was.
        ea, eb, _pa, _pb, _groups, seasons_arr = _pair(market, "baseline", arm)
        for season in VALID_SEASONS:
            sel = seasons_arr == season
            if sel.sum() == 0:
                continue
            d = float(eb[sel].mean() - ea[sel].mean())   # treated - baseline, negative = better
            stab.append({"market": market, "arm": arm, "season": int(season),
                         "delta_mae": d, "helped": bool(d < 0), "n": int(sel.sum())})
stability = pd.DataFrame(stab)
stability_summary = (stability.groupby(["market", "arm"])
                     .agg(seasons=("season", "nunique"), seasons_helped=("helped", "sum"),
                          mean_delta=("delta_mae", "mean"), worst_delta=("delta_mae", "max"),
                          best_delta=("delta_mae", "min"),
                          sign_stable=("helped", lambda s: bool(len(set(s.tolist())) == 1)))
                     .reset_index())
stability_summary["share_helped"] = (stability_summary.seasons_helped
                                     / stability_summary.seasons).round(3)
stability_summary["stable"] = stability_summary.sign_stable & (stability_summary.share_helped >= 0.6)
print()
display(stability_summary.round(4))

fig, ax = plt.subplots(figsize=(9.5, 4.2))
piv = (stability[stability.arm == "plus_ratings_z"]
       .pivot(index="market", columns="season", values="delta_mae").reindex(PRIMARY_MARKETS))
M = piv.to_numpy(float); lim = np.nanmax(np.abs(M))
im = ax.imshow(M, cmap="RdYlGn_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if np.isfinite(M[i, j]):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Delta MAE by test season (green = ratings helped). A sign flip = unstable.")
plt.colorbar(im, ax=ax, label="delta MAE"); plt.tight_layout(); plt.show()

In [ ]:
# ------------------------------------------------------------------ A.5 VIF, variants
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

d = rated[ARMS["plus_ratings_z"]].fillna(0).astype(float).to_numpy()
vif_table = pd.DataFrame([{"feature": c, "VIF": float(vif(d, i))}
                          for i, c in enumerate(ARMS["plus_ratings_z"])])
print("VIF for the treated design (baseline + four z-ratings). Above ~10 is conventionally")
print("'severe'. The pass and rush ratings are structurally correlated, which is exactly why")
print("A.5 also asks for a combined rating and an orthogonalised pass-minus-rush.")
display(vif_table.sort_values("VIF", ascending=False).round(2))
print("\ncorrelation among the four z-ratings:")
display(rated[RATING_Z].corr().round(3))

rated["pass_rush_combined"] = (rated.pass_off_z + rated.rush_off_z) / 2.0
rated["def_pass_rush_combined"] = (rated.pass_def_z + rated.rush_def_z) / 2.0
_p, _r = rated.pass_off_z - rated.pass_def_z, rated.rush_off_z - rated.rush_def_z
rated["pass_minus_rush_orth"] = ((_p - _p.mean()) / _p.std()) - ((_r - _r.mean()) / _r.std())

ALT_ARMS = {"combined_rating": ARMS["baseline"] + ["pass_rush_combined", "def_pass_rush_combined"],
            "orthogonalised":   ARMS["baseline"] + ["pass_minus_rush_orth"]}
alt_rows = []
for market in ["rushing_yards", "receiving_yards"]:
    base = next(x for x in METRICS if x["market"] == market and x["arm"] == "baseline")
    for arm, ccols in ALT_ARMS.items():
        m = arm_metrics(score_arm(rated, market, ccols), market)
        alt_rows.append({"market": market, "arm": arm, "baseline_mae": base["mae_q50"],
                         "mae_q50": m["mae_q50"], "delta_mae": m["mae_q50"] - base["mae_q50"]})
alt_arms = pd.DataFrame(alt_rows)
print("\nPart A.5 second ask: pass/rush combined into one rating, and orthogonalised pass-minus-rush")
display(alt_arms.round(4))

In [ ]:
# ------------------------------------------------------------------ A.5 multiple testing (BH)
def benjamini_hochberg(p):
    p = np.asarray(p, float); n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    out = np.empty(n)
    out[order] = np.minimum.accumulate(ranked[::-1])[::-1]
    return np.clip(out, 0, 1)

F1 = bootstrap.copy()
print(f"TESTS RUN in Part A, family F1 (ratings vs baseline): {len(F1)}.")
print("Counted up front, BEFORE looking at which survived -- that is the point of BH.")
F1["p_value"] = F1.p_bootstrap
F1["p_bh"] = benjamini_hochberg(F1.p_value.to_numpy())
F1["bh_survivor_05"] = F1.p_bh < 0.05
F1["ci_excludes_decline"] = F1.delta_mae_hi < 0
cal_idx = calibration.set_index(["market", "arm"])
F1["calib_gap_widened"] = [bool(cal_idx.loc[(m, a), "worst_gap"]
                               > cal_idx.loc[(m, "baseline"), "worst_gap"])
                           for m, a in zip(F1.market, F1.arm)]
F1["stable"] = [bool(stability_summary.set_index(["market", "arm"]).loc[(m, a), "stable"])
                for m, a in zip(F1.market, F1.arm)]

SHOW = ["market", "arm", "delta_mae", "delta_mae_lo", "delta_mae_hi", "p_value", "p_bh",
        "bh_survivor_05", "ci_excludes_decline", "calib_gap_widened", "stable"]
print("\nBENJAMINI-HOCHBERG across all F1 tests:")
display(F1[SHOW].round(4))

print("\nTHE SHIP GATE, stated once: a candidate ships only if the paired-bootstrap CI EXCLUDES A")
print("DECLINE **and** the calibration gap does not widen. Both, or neither. Nothing is lowered.")
F1["ship_gate_passes"] = F1.ci_excludes_decline & ~F1.calib_gap_widened
display(F1[SHOW + ["ship_gate_passes"]].round(4))
print(f"tests clearing the ship gate: {int(F1.ship_gate_passes.sum())} of {len(F1)}")
F1.to_csv(STUDY_CACHE / "partA_f1_bh.csv", index=False)

In [ ]:
# ------------------------------------------------------------------ A.6 placebo
print("Placebo: opponent ratings shuffled across teams within the same (season, week), so each")
print("rating keeps its distribution and stays plausible for that week but loses all meaning.")
print("Same harness, same folds, same rows. Only the ratings become noise.")

def placebo_frame(frame, cols, seed):
    rng = np.random.default_rng(seed)
    out = frame.copy()
    blocks = frame.groupby(["season", "week"]).indices
    for c in cols:
        vals = frame[c].to_numpy(float).copy()
        for _, idx in blocks.items():
            v = vals[idx]; ok = np.isfinite(v)
            if ok.sum() > 1:
                vals[idx[ok]] = rng.permutation(v[ok])
        out[c] = vals
    return out

placebo = placebo_frame(rated, RATING_Z, seed=987654321)
pl_rows = []
for market in ["passing_yards", "rushing_yards", "receiving_yards"]:
    m = arm_metrics(score_arm(placebo, market, ARMS["plus_ratings_z"]), market)
    base = next(x for x in METRICS if x["market"] == market and x["arm"] == "baseline")
    real = next(x for x in BOOT if x["market"] == market and x["arm"] == "plus_ratings_z")
    pl_rows.append({"market": market, "baseline_mae": base["mae_q50"],
                    "real_delta_mae": real["delta_mae"], "placebo_delta_mae": m["mae_q50"] - base["mae_q50"]})
placebo_table = pd.DataFrame(pl_rows)
# Ratio of the placebo effect to the real one. Only meaningful where the real effect is
# measurable at all: where it is ~0, "half of it" is ~0 and the ratio explodes on noise. The
# floor below is stated, not chosen to make this pass.
FLOOR = 0.05          # yards of real effect; below this the market is declared a null
placebo_table["ratio"] = np.where(placebo_table.real_delta_mae.abs() >= FLOOR,
                                  placebo_table.placebo_delta_mae.abs() / placebo_table.real_delta_mae.abs(),
                                  np.nan)
placebo_table["real_effect_measurable"] = placebo_table.real_delta_mae.abs() >= FLOOR
display(placebo_table.round(4))
_meas = placebo_table[placebo_table.real_effect_measurable]
collapsed = bool((_meas.ratio <= 0.5).all()) if len(_meas) else False
print(f"markets where the real effect is measurable (>= {FLOOR} yards): {len(_meas)} of {len(placebo_table)}")
print(f"placebo/real ratio on those markets: {list(_meas.ratio.round(3))}  (all <= 0.5: {collapsed})")
print("On the markets with NO measurable real effect the placebo is also ~0, which is the")
print("correct outcome -- there was nothing for it to reproduce.")
print("(A placebo that DID reproduce the real gain would mean the gain came from adding four")
print(" more columns to the design, not from the information inside them.)")

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(placebo_table)); w = 0.35
ax.bar(x - w/2, placebo_table.real_delta_mae, w, label="real ratings", color="#4C72B0")
ax.bar(x + w/2, placebo_table.placebo_delta_mae, w, label="placebo (shuffled)", color="#B0B0B0")
ax.axhline(0, color="black", lw=1); ax.set_xticks(x); ax.set_xticklabels(placebo_table.market)
ax.set_ylabel("delta MAE vs baseline"); ax.set_title("Part A.6  Placebo test")
ax.legend(); ax.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.show()

## Receptions through the repo's own ZINB (in-memory patch, disclosed)

**Plain English.** Receptions are a count market gated by `receptions_zinb`, not by the quantile
regressor, and that module hard-codes its own nine columns. Reaching it with the ratings would
mean editing the file, which this read-only study may not do. So the ZINB's likelihood, line
function and gate arithmetic are kept exactly as they are, and only `_derive_features` is
wrapped **in memory** to append the rating columns to the feature row. Nothing on disk changes.
The caveat travels with the number: this is the repo's gate applied to an augmented design, not
a gate the repo itself would produce today.

In [ ]:
_ORIG_DERIVE = receptions_zinb._derive_features

def _derive_with_ratings(df):
    """The repo's own 9 features, then the ratings appended.

    `RECEPTIONS_FEATURE_COLUMNS` is temporarily restored to the original 9 while
    `_ORIG_DERIVE` runs, because `_derive_features` ends with `out[list(RECEPTIONS_FEATURE_COLUMNS)]`
    and would raise KeyError on the rating names the moment the extended list is installed. The
    ZINB's parameter count comes from the returned matrix's width (`k = Xs.shape[1]`), not from
    the constant, so nothing about the likelihood, the line function or the gate changes."""
    saved = receptions_zinb.RECEPTIONS_FEATURE_COLUMNS
    receptions_zinb.RECEPTIONS_FEATURE_COLUMNS = tuple(ZINB_COLS)
    try:
        base = _ORIG_DERIVE(df)
    finally:
        receptions_zinb.RECEPTIONS_FEATURE_COLUMNS = saved
    extra = [c for c in RATING_Z if c in df.columns]
    for c in extra:
        base[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    return base

def receptions_zinb_arm(frame, ratings_enabled, seasons=(2022, 2023, 2024)):
    """The repo's own ZINB walk-forward + gate, with ratings appended if asked.

    `RECEPTIONS_FEATURE_COLUMNS` and `_derive_features` are restored in a `finally`, so a later
    cell sees the untouched module even if this one raises."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        receptions_zinb.RECEPTIONS_FEATURE_COLUMNS = tuple(ZINB_COLS) + tuple(
            RATING_Z if ratings_enabled else ())
        receptions_zinb._derive_features = _derive_with_ratings if ratings_enabled else _ORIG_DERIVE
        try:
            sc = receptions_zinb.walk_forward_receptions(frame, list(seasons))
            if sc.empty:
                return {}, sc
            return receptions_zinb.receptions_calibration_gate(sc), sc
        except RuntimeError as exc:
            # The ZINB's own optimiser refuses the augmented design. Reported, not worked
            # around: the model is unregularised (a log link on every coefficient, no penalty),
            # so its feature budget is already spent at 9 columns. Shrinking the rating set to
            # whatever fits would mean choosing the measurement after seeing which one
            # converged, which is exactly what the pre-statement rule forbids.
            print(f"    ZINB arm '{'plus_ratings_z' if ratings_enabled else 'baseline'}' did not "
                  f"converge: {exc}")
            return None, pd.DataFrame()
        finally:
            receptions_zinb.RECEPTIONS_FEATURE_COLUMNS = tuple(ZINB_COLS)
            receptions_zinb._derive_features = _ORIG_DERIVE

zinb_rows = []
for arm, enabled in (("baseline", False), ("plus_ratings_z", True)):
    gate, sc = receptions_zinb_arm(rated, enabled)
    if gate is None:
        zinb_rows.append({"market": "receptions (ZINB)", "arm": arm, "n": 0,
                          "features": len(ZINB_COLS) + (len(RATING_Z) if enabled else 0),
                          "buckets": 0, "worst_gap": np.nan, "gate_passes": False,
                          "failing_buckets": "DID NOT CONVERGE -- not computable"})
        continue
    failing = sorted(k for k, v in gate.items() if not v.get("within_tolerance", False))
    zinb_rows.append({"market": "receptions (ZINB)", "arm": arm, "n": int(len(sc)),
                      "features": len(ZINB_COLS) + (len(RATING_Z) if enabled else 0),
                      "buckets": len(gate),
                      "worst_gap": max((v["gap"] for v in gate.values()), default=float("nan")),
                      "gate_passes": calibration_passes(gate),
                      "failing_buckets": ",".join(failing) or "(none)"})
receptions_zinb_result = pd.DataFrame(zinb_rows)
print("WR/TE RECEPTIONS -- the repo's ZINB gate (10 bins, tol 0.05, n>=100, not lowered)")
display(receptions_zinb_result.round(4))
print("Repo note carried forward: RB receptions are deliberately unpriced (the ZINB fails on")
print("backs), and that stays unpriced here -- no RB receptions row is scored or claimed.")
print()
print("Reading: the repo's own ZINB is AT ITS NUMERICAL LIMIT at 9 columns. Adding four")
print("causal, well-behaved (VIF ~1.0-1.9) ratings exhausts its unregularised optimiser, so the")
print("receptions-with-ratings question is NOT COMPUTABLE with this model as written. That is a")
print("finding about the model, not about the ratings: the count model has no headroom for more")
print("inputs without a penalty term. Receptions verdict: NOTHING (not measured, not refuted).")

# Part B — game script: predict how the game goes, then price from the script

**Plain English.** Part A asked whether four team ratings help a model that predicts a player's
yardage in isolation. Part B asks the question the product is actually heading towards: **can we
predict the shape of the game before kickoff, and does that shape explain a player's production
better than the player's own recent numbers do?** Four steps, in order. First, predict the
team-level quantities that define a script — pass-attempt share, rush attempts, plays, points —
from pre-game information only, and report how much of that is explained by spread and total
alone versus by our ratings. Second, convert the projected team volume into player volume
(projected pass attempts × the player's target share) and test whether that beats raw recent
usage. Third, the spillover question — does a *pre-game projected* RB carry count change a
receiver's targets — with the reverse-causality trap handled by never using a realised carry.
Fourth, price: does conditioning the quantile band on script state fix P(over) calibration in
any bucket.

Everything here is pre-game information only. `spread_line`/`total_line` are nflverse **closing**
lines, which `training.py` excludes from the baseline for exactly this train/serve skew, so every
spread/total number in this part is an **upper bound** on what a pre-kickoff snapshot could use.

In [ ]:
# ------------------------------------------------------------------ B.1 team-level script quantities
SCRIPT_OUTCOMES = {
    "pass_attempt_share": None,   # team pass attempts / (pass + rush attempts)
    "rush_attempts": "rush_attempts",
    "plays": "plays",
    "points": "points",
}

def team_game_frame():
    """One row per team-game with pre-game inputs and the script outcomes."""
    tg = build_team_games().copy()
    # Pre-game inputs, read off the player frame (one row per team-game there).
    # `is_home` is NOT taken from the player frame: build_team_games already derives it from
    # home_team, and merging a second copy produces is_home_x/is_home_y and every later
    # reference to `is_home` then raises KeyError.
    pg_in = analysis.groupby(["season", "week", "recent_team"]).agg(
        spread_line=("spread_line", "first"), total_line=("total_line", "first"),
        rest_days=("rest_days", "first"), temp_c=("temp_c", "first"),
        wind_kph=("wind_kph", "first"),
        implied_team_total=("implied_team_total", "first")).reset_index().rename(
        columns={"recent_team": "team"})
    out = tg.merge(pg_in, on=["season", "week", "team"], how="left")
    out["pass_attempt_share"] = out.pass_attempts / (out.pass_attempts + out.rush_attempts)
    # The PREGAME EL0 rating is already causal (power_ratings.compute_pregame_ratings is
    # chronological), so joining it here adds no lookahead.
    out = out.merge(ratings[["season", "week", "recent_team", "pass_off", "pass_def",
                             "rush_off", "rush_def", "pass_off_z", "pass_def_z",
                             "rush_off_z", "rush_def_z"]]
                    .rename(columns={"recent_team": "team"}),
                    on=["season", "week", "team"], how="left")
    return out.sort_values(["season", "team", "week"]).reset_index(drop=True)

TGX = team_game_frame()
TGX["season"] = TGX.season.astype(int)
SCRIPT_BASE = ["spread_line", "total_line", "is_home", "rest_days", "temp_c", "wind_kph",
               "elo_diff"]
SCRIPT_RATED = SCRIPT_BASE + [f"{s}_z" for s in RATING_STATS]

def walk_forward_team(df, outcome, cols, seasons=VALID_SEASONS):
    """Season-by-season: fit on seasons < S, predict S. Reports out-of-sample MAE and R^2."""
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.linear_model import LinearRegression
    rows = []
    for val_season in seasons:
        tr = df[df.season < val_season].dropna(subset=[outcome])
        te = df[(df.season == val_season)].dropna(subset=[outcome])
        if tr.empty or te.empty:
            continue
        Xtr = tr[cols].astype(float).fillna(0.0); Xte = te[cols].astype(float).fillna(0.0)
        mu_tr, sd_tr = Xtr.mean(), Xtr.std().replace(0, 1.0)
        Xtr, Xte = (Xtr - mu_tr) / sd_tr, (Xte - mu_tr) / sd_tr
        # linear and gradient-boosted both, so a non-linear script relationship is not missed
        for name, model in (("linear", LinearRegression()),
                            ("gbt", GradientBoostingRegressor(random_state=42, n_estimators=120,
                                                             max_depth=3, learning_rate=0.05))):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model.fit(Xtr, tr[outcome].to_numpy(float))
            p = model.predict(Xte)
            y = te[outcome].to_numpy(float)
            rows.append({"outcome": outcome, "model": name, "season": int(val_season),
                         "n": int(len(y)), "mae": float(np.abs(y - p).mean()),
                         "r2": float(1 - ((y - p) ** 2).sum() / max(((y - y.mean()) ** 2).sum(), 1e-9)),
                         "naive_mae": float(np.abs(y - y.mean()).mean())})
    return pd.DataFrame(rows)

script_rows = []
for outcome in SCRIPT_OUTCOMES:
    for tag, cols in (("pregame_only", SCRIPT_BASE), ("plus_ratings", SCRIPT_RATED)):
        for model in ("linear", "gbt"):
            r = walk_forward_team(TGX, outcome, cols)
            if r.empty:
                continue
            r = r[r.model == model]
            for _, x in r.iterrows():
                script_rows.append({"outcome": outcome, "features": tag, "model": model,
                                    "season": x.season, "n": x.n, "mae": x.mae, "r2": x.r2,
                                    "naive_mae": x.naive_mae})
script_folds = pd.DataFrame(script_rows)
print("B.1 TEAM-LEVEL SCRIPT QUANTITIES -- out-of-sample, pre-game inputs only")
display(script_folds.groupby(["outcome", "features", "model"])[
    ["mae", "r2", "naive_mae", "n"]].mean().round(4))
print("Negative R^2 means worse than predicting the pooled mean. spread+total are CLOSING")
print("lines here, so read every number as an upper bound.")

piv = script_folds.groupby(["outcome", "features", "model"]).r2.mean().unstack(["features", "model"])
display(piv.round(4))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.9))
for ax, outcome in zip(axes, SCRIPT_OUTCOMES):
    sub = script_folds[script_folds.outcome == outcome].groupby(
        ["features", "model", "season"]).mae.mean().reset_index()
    for (feat, model), g in sub.groupby(["features", "model"]):
        ax.plot(g.season, g.mae, marker="o", label=f"{feat}/{model}")
    ax.set_title(outcome); ax.set_xlabel("test season"); ax.set_ylabel("out-of-sample MAE")
    ax.grid(alpha=.3)
axes[0].legend(fontsize=7); fig.suptitle("Part B.1  Predicting the game, out of sample")
plt.tight_layout(); plt.show()

In [ ]:
# ------------------------------------------------------------------ B.2 script-implied volume
# Does the player's own recent usage, or the script, explain his production better?
# Script-implied volume = projected team attempts x the player's lagged share of that volume.
print("B.2 SCRIPT vs RAW USAGE")
print("Projected volume is built from the B.1 projections evaluated on every row, plus the")
print("player's LAGGED share of his team's recent volume. Never a same-week share.")

script_pred = {}
for outcome in ("pass_attempts", "rush_attempts", "plays"):
    r = walk_forward_team(TGX, outcome, SCRIPT_RATED, seasons=VALID_SEASONS)
    if r.empty:
        continue
    from sklearn.ensemble import GradientBoostingRegressor
    for val_season in VALID_SEASONS:
        tr = TGX[TGX.season < val_season].dropna(subset=[outcome])
        te = TGX[TGX.season == val_season].dropna(subset=[outcome])
        if tr.empty or te.empty:
            continue
        Xtr = tr[SCRIPT_RATED].astype(float).fillna(0.0)
        Xte = te[SCRIPT_RATED].astype(float).fillna(0.0)
        mu, sd = Xtr.mean(), Xtr.std().replace(0, 1.0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = GradientBoostingRegressor(random_state=42, n_estimators=120, max_depth=3,
                                          learning_rate=0.05).fit((Xtr - mu) / sd,
                                                                  tr[outcome].to_numpy(float))
        pred = m.predict((Xte - mu) / sd)
        script_pred[(outcome, int(val_season))] = pd.Series(
            pred, index=te.set_index(["season", "week", "team"]).index)

# The dict keys become index levels named ["outcome", "season"], and the Series inside already
# has a level called "season" -- two levels with one name, and reset_index then raises
# "cannot insert season, already exists". The key level is therefore named `pred_season`.
tgp = pd.concat({k: v for k, v in script_pred.items()},
                names=["outcome", "pred_season"]).reset_index()
tgp = tgp.rename(columns={"season": "season", "team": "recent_team", 0: "proj"})
SCRIPT_VOLS = tgp.pivot_table(index=["season", "week", "recent_team"],
                              columns="outcome", values="proj").reset_index()
# the pivot's column labels are the bare outcome names; prefix them so a projected volume is
# never confused with the REALISED team volume column carried in the frame
SCRIPT_VOLS.columns = [f"proj_{c}" if c in ("pass_attempts", "rush_attempts", "plays") else c
                       for c in SCRIPT_VOLS.columns]
print("projected team volumes:", SCRIPT_VOLS.shape)
display(SCRIPT_VOLS.head(3))

b2 = rated.merge(SCRIPT_VOLS, on=["season", "week", "recent_team"], how="left")
# LAGGED shares: the player's own share of his team's volume in PRIOR games.
ps = b2.sort_values(["player_id", "season", "week"])
for num, den, name in (("targets", "proj_pass_attempts", "share_targets"),
                       ("carries", "proj_rush_attempts", "share_carries"),
                       ("attempts", "proj_pass_attempts", "share_attempts")):
    if num in ps.columns:
        ps[name] = ps.groupby("player_id")[num].transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).mean())
ps["implied_targets"] = ps.share_targets * ps.proj_pass_attempts
ps["implied_carries"] = ps.share_carries * ps.proj_rush_attempts
ps["implied_attempts"] = ps.share_attempts * ps.proj_pass_attempts
print("\nscript-implied volumes available:", [c for c in
      ("implied_targets", "implied_carries", "implied_attempts") if c in ps.columns])

# market -> (the player's OWN usage column the baseline leans on, the script-implied column)
# The three arms must genuinely differ, which the first version did NOT: `augment_baseline`
# was byte-identical to `baseline` on every market, so it reported a delta of exactly 0.0 and
# proved nothing. The three arms are:
#   baseline          the 14 served columns (own lagged usage present)
#   augment_baseline  baseline + script-implied volume
#   replace_usage     baseline MINUS the player's own usage roll, PLUS script-implied volume
VOL_MARKETS = {"attempts":         ("attempts_roll", "implied_attempts", "QB"),
               "passing_yards":   ("passing_yards_roll", "implied_attempts", "QB"),
               "carries":         ("carries_roll", "implied_carries", "RB"),
               "rushing_yards":   ("rushing_yards_roll", "implied_carries", "RB"),
               "targets":         ("targets_roll", "implied_targets", "WR/TE"),
               "receiving_yards": ("receiving_yards_roll", "implied_targets", "WR/TE")}

b2_rows = []
for market, (usage_col, script_col, pos) in VOL_MARKETS.items():
    for arm in ("baseline", "augment_baseline", "replace_usage"):
        if script_col not in ps.columns:
            continue
        if arm == "augment_baseline":
            cols = list(ARMS["baseline"]) + [script_col]
        elif arm == "replace_usage":
            cols = [c for c in ARMS["baseline"] if c != usage_col] + [script_col]
        else:
            cols = list(ARMS["baseline"])
        m = arm_metrics(score_arm(ps, market, cols), market)
        b2_rows.append({"market": market, "position": pos, "arm": arm, "features": len(cols),
                        "n": m["n"], "mae_q50": m["mae_q50"], "pin_q50": m["pin_q50"]})
b2_metrics = pd.DataFrame(b2_rows)
print("\nB.2 script-implied volume vs the player's own recent usage")
display(b2_metrics.pivot(index=["market", "position"], columns="arm",
                         values=["mae_q50", "pin_q50"]).round(4))
b2_delta = (b2_metrics.pivot(index=["market", "position"], columns="arm", values="mae_q50"))
b2_delta["delta_augment"] = b2_delta["augment_baseline"] - b2_delta["baseline"]
b2_delta["delta_replace"] = b2_delta["replace_usage"] - b2_delta["baseline"]
print("\ndelta MAE (negative = script-implied volume helps). Verdict per position below.")
display(b2_delta.round(4))

### B.3 Spillover, with the reverse-causality trap handled

**Plain English.** The naive version of this question is invalid and the notebook refuses to run
it. Game script drives both RB carries and WR targets, so regressing a receiver's targets on the
running back's **realised** carries measures how the game went, not what the script predicted.
Everything below uses **pre-game projected** RB1 carries, produced by the same B.1 projection
that will exist at serve time, and holds projected team pass attempts fixed. That converts a
confounded descriptive question into a causal one: *holding the pass volume the projection
expects, would a team projected to run more also throw more to the secondary receiver?*

In [ ]:
# ------------------------------------------------------------------ B.3 spillover
import statsmodels.formula.api as smf

# Pre-game projected RB carries: projected rush attempts x the RB's LAGGED carry share.
rb = ps[ps.position == "RB"].copy()
# A SHARE, not a count. Multiplying the player's lagged carries-per-game (about 15) by the
# team's projected rush attempts (about 25) produced a "projected carry" of ~375, which is not a
# carry count and made every downstream marginal effect unreadable. The share is the player's
# lagged carries over his team's lagged total carries -- still entirely pre-game.
_team_carries = (rb.groupby(["season", "week", "recent_team"]).carries.transform("sum")
                 .groupby(rb.player_id).transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean()))
rb["carry_share_lag"] = (rb.groupby("player_id")["carries"]
                         .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
                         / _team_carries.replace(0, np.nan))
rb["carry_share_lag"] = rb.carry_share_lag.clip(0, 1).fillna(0.0)
rb["proj_carries"] = rb.carry_share_lag * rb.proj_rush_attempts
# RB1 = the team's highest projected-carry back in that game.
rb1 = (rb.sort_values(["season", "week", "recent_team", "proj_carries"],
                      ascending=[True, True, True, False])
       .drop_duplicates(["season", "week", "recent_team"])[
           ["season", "week", "recent_team", "player_id", "proj_carries"]]
       .rename(columns={"player_id": "rb1_id", "proj_carries": "proj_rb1_carries"}))
print("B.3 SPILLOVER -- pre-game projected RB1 carries, never realised carries")
print(f"RB1 games identified: {len(rb1):,} across {rb1.groupby(['season','week']).ngroups:,} team-games")

# Receiver rank within the game by lagged target share, so WR1/WR2/TE are defined PRE-GAME.
wr = ps[ps.position.isin(["WR", "TE"])].copy()
wr["target_share_lag"] = wr.groupby("player_id")["targets"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean())
wr["rec_rank"] = wr.sort_values(["season", "week", "recent_team", "target_share_lag"],
                                ascending=[True, True, True, False]) \
                 .groupby(["season", "week", "recent_team"]).cumcount() + 1
wr = wr.merge(rb1, on=["season", "week", "recent_team"], how="left")
wr["is_rb1"] = wr.player_id == wr.rb1_id
wr["wr_slot"] = np.where(wr.is_rb1, "RB1(not a receiver)",
                         np.where(wr.rec_rank == 1, "WR1",
                                  np.where(wr.rec_rank == 2, "WR2",
                                           np.where(wr.rec_rank == 3, "WR3", "WR4+"))))
spill = wr[wr.wr_slot.isin(["WR1", "WR2", "WR3"])].copy()
print("receiver rows in the spillover frame:", spill.shape[0])
print(spill.wr_slot.value_counts().to_dict())

CTRL = ["proj_rb1_carries", "proj_pass_attempts", "spread_line", "total_line", "is_home"]
CONTROLS = [c for c in CTRL if c in spill.columns] + ["target_share_lag"]
MODELS = {}
def fit_spill(outcome, extra=()):
    f = spill.dropna(subset=[outcome] + [c for c in CONTROLS if c in spill.columns]).copy()
    if len(f) < 500:
        return None
    formula = f"{outcome} ~ " + " + ".join(CONTROLS + list(extra))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.ols(formula, f).fit(cov_type="cluster",
                                    cov_kwds={"groups": f.player_id})
    return m, f

print("\nDoes PRE-GAME PROJECTED RB1 CARRY volume change a receiver's target share and yards,")
print("holding projected team pass attempts, spread, total and home/away fixed?")
sp_rows = []
for outcome in ("targets", "receiving_yards"):
    got = fit_spill(outcome)
    if got is None:
        continue
    m, f = got
    b = m.params.get("proj_rb1_carries", np.nan)
    se = m.bse.get("proj_rb1_carries", np.nan)
    ci = m.conf_int().loc["proj_rb1_carries"]
    sp_rows.append({"outcome": outcome, "n": int(len(f)),
                    "effect_per_1_extra_rush_attempt": b, "se": se,
                    "ci_lo": ci[0], "ci_hi": ci[1],
                    "effect_per_10_more_carries": 10 * b})
spill_main = pd.DataFrame(sp_rows)
display(spill_main.round(5))
print("coefficient units: outcome units per ONE more projected rush attempt by the projected RB1.")
print("'10 more carries' is the readable version; it is a linear extrapolation and is labelled as such.")

# per-slot marginal effects, low / mid / high
print("\nMarginal effects by receiver slot (low / mid / high projected RB1 volume), with n per cell.")
mfx = []
for slot in ("WR1", "WR2", "WR3"):
    for outcome in ("targets", "receiving_yards"):
        sub = spill[spill.wr_slot == slot]
        f = sub.dropna(subset=[outcome] + [c for c in CONTROLS if c in sub.columns]).copy()
        if len(f) < 200:
            continue
        formula = f"{outcome} ~ " + " + ".join(CONTROLS)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = smf.ols(formula, f).fit(cov_type="cluster", cov_kwds={"groups": f.player_id})
        # low / mid / high = terciles of projected RB1 carries WITHIN this slot. Bins, not
        # `np.isclose` on a continuous value: that matched 0-2 rows per cell and produced a
        # table of empty marginal effects.
        b = m.params.get("proj_rb1_carries", np.nan)
        edges = f.proj_rb1_carries.quantile([0.0, 1 / 3, 2 / 3, 1.0]).tolist()
        f = f.assign(_cell=pd.cut(f.proj_rb1_carries, edges, labels=["low", "mid", "high"],
                                  include_lowest=True))
        for lbl in ("low", "mid", "high"):
            cell = f[f._cell == lbl]
            mid = float(cell.proj_rb1_carries.mean()) if len(cell) else np.nan
            # predict on a real row frame at the cell's mean projected carries, and ravel the
            # result: statsmodels hands back a Series indexed by ROW LABEL, so positional [0]
            # raises KeyError.
            if len(cell):
                row = cell.iloc[[0]].copy()
                row["proj_rb1_carries"] = mid
                pred = float(np.asarray(m.predict(row)).ravel()[0])
            else:
                pred = np.nan
            mfx.append({"slot": slot, "outcome": outcome, "level": lbl,
                        "proj_rb1_carries": mid, "n": int(len(cell)), "predicted": pred})
        mfx.append({"slot": slot, "outcome": outcome, "level": "slope_per_10",
                    "proj_rb1_carries": np.nan, "n": int(len(f)), "predicted": 10 * b})
spill_marg = pd.DataFrame(mfx)
display(spill_marg.round(4))

# ---- does QB projected pass volume change WR2 / WR3 outcomes? -------------------------
print("\nDoes projected QB PASS ATTEMPTS (or pass TDs) change the SECOND receiver?")
qb_rows = []
for outcome in ("targets", "receiving_yards"):
    for slot in ("WR1", "WR2", "WR3"):
        sub = spill[spill.wr_slot == slot]
        cols = [c for c in ("proj_pass_attempts", "spread_line", "total_line", "is_home",
                            "target_share_lag", "proj_rb1_carries") if c in sub.columns]
        f = sub.dropna(subset=[outcome] + cols).copy()
        if len(f) < 200:
            continue
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = smf.ols(f"{outcome} ~ " + " + ".join(cols), f).fit(
                cov_type="cluster", cov_kwds={"groups": f.player_id})
        ci = m.conf_int().loc["proj_pass_attempts"]
        qb_rows.append({"slot": slot, "outcome": outcome, "n": int(len(f)),
                        "effect_per_extra_pass_attempt": m.params["proj_pass_attempts"],
                        "ci_lo": ci[0], "ci_hi": ci[1]})
qb_spill = pd.DataFrame(qb_rows)
display(qb_spill.round(5))

# ---- interaction: projected RB carries x opponent pass-defence rating ----------------
print("\nINTERACTION: projected RB carries x opponent pass-defence rating")
print("On RB rushing yards, and on WR/TE receiving yards.")
inter = []
# `ps` already carries pass_def_z (it descends from `rated`), so re-merging the ratings table
# here would produce pass_def_z_x/_y and the formula below would fail on a missing name.
rbj = rb.copy()
rbj["z_carries"] = (rbj.proj_carries - rbj.proj_carries.mean()) / rbj.proj_carries.std()
cols = ["z_carries", "pass_def_z", "spread_line", "total_line", "is_home", "carries_roll"]
for outcome in ("rushing_yards",):
    f = rbj.dropna(subset=[outcome] + [c for c in cols if c in rbj.columns]).copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.ols(f"{outcome} ~ z_carries * pass_def_z + " +
                    " + ".join(c for c in cols[3:]), f).fit(
            cov_type="cluster", cov_kwds={"groups": f.player_id})
    ci = m.conf_int().loc["z_carries:pass_def_z"]
    inter.append({"sample": "RB", "outcome": outcome, "n": int(len(f)),
                  "interaction": m.params["z_carries:pass_def_z"],
                  "ci_lo": ci[0], "ci_hi": ci[1]})
for slot in ("WR1", "WR2"):
    sub = spill[spill.wr_slot == slot].copy()
    sub["z_carries"] = (sub.proj_rb1_carries - sub.proj_rb1_carries.mean()) / sub.proj_rb1_carries.std()
    cols = ["z_carries", "pass_def_z", "target_share_lag", "spread_line", "total_line"]
    f = sub.dropna(subset=["receiving_yards"] + [c for c in cols if c in sub.columns]).copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.ols("receiving_yards ~ z_carries * pass_def_z + target_share_lag + spread_line"
                    " + total_line", f).fit(cov_type="cluster", cov_kwds={"groups": f.player_id})
    ci = m.conf_int().loc["z_carries:pass_def_z"]
    inter.append({"sample": slot, "outcome": "receiving_yards", "n": int(len(f)),
                  "interaction": m.params["z_carries:pass_def_z"],
                  "ci_lo": ci[0], "ci_hi": ci[1]})
interaction = pd.DataFrame(inter)
display(interaction.round(5))

fig, ax = plt.subplots(figsize=(9, 4.2))
if len(spill_marg):
    sub = spill_marg[spill_marg.outcome == "receiving_yards"].copy()
    for slot, g in sub.groupby("slot"):
        g = g[g.level != "slope_per_10"]
        ax.plot(g.proj_rb1_carries, g.predicted, marker="o", label=slot)
    ax.set_xlabel("pre-game projected RB1 carries"); ax.set_ylabel("predicted receiving yards")
    ax.set_title("Part B.3  Spillover: receiver output vs PROJECTED RB1 volume\n"
                 "(reverse causality avoided -- no realised carry is ever an input)")
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ------------------------------------------------------------------ B.4 pricing from script state
print("B.4 BETTING ANGLE -- does conditioning the quantile band on script state fix P(over)?")
print("Calibration is the repo's own: calibration_report, 10 bins, tolerance 0.05, n >= 100.")
print("Tolerance is NOT lowered anywhere, and buckets below n=100 are omitted, not reported.")

def script_bucket(df):
    """Pre-game script state, three cuts chosen from the pre-game inputs only."""
    side = 2 * df.is_home.fillna(0) - 1
    margin = side * df.spread_line                 # positive = this team is favoured
    d = pd.DataFrame(index=df.index)
    d["favourite"] = pd.cut(margin, [-np.inf, -3, 3, np.inf], labels=["underdog", "pickem", "favourite"])
    d["total_bucket"] = pd.cut(df.total_line, [-np.inf, 42, 46, np.inf],
                               labels=["low_total", "mid_total", "high_total"])
    return d

def calibration_by_bucket(scored, bucket_labels, n_bins=10, min_n=100):
    """The repo's calibration_report applied inside each bucket. Same arithmetic, sliced."""
    rows = []
    joined = scored.copy()
    for col, vals in bucket_labels.items():
        joined[col] = vals.reindex(joined.index).to_numpy()
    for bkey, g in joined.groupby(list(bucket_labels.keys()), observed=True):
        rep = calibration_report(g, columns=("p_over_exogenous", "covered_exogenous"))
        if not rep:
            rows.append({"bucket": str(bkey), "buckets": 0, "n": int(len(g)),
                         "worst_gap": np.nan, "mean_pred": np.nan, "mean_emp": np.nan})
            continue
        rows.append({"bucket": str(bkey), "buckets": len(rep), "n": int(len(g)),
                     "worst_gap": max(b["gap"] for b in rep.values()),
                     "mean_pred": float(np.nanmean([b["predicted"] for b in rep.values()])),
                     "mean_emp": float(np.nanmean([b["empirical"] for b in rep.values()]))})
    return pd.DataFrame(rows)

base_scored = pd.concat([SCORED[(m, "baseline")].assign(market=m) for m in PRIMARY_MARKETS],
                        ignore_index=True)
# script state has to be attached by (season, week, team) and then broadcast to player rows
state = TGX[["season", "week", "team", "spread_line", "total_line", "is_home"]].copy()
labels = script_bucket(state)
state = state.assign(**{c: labels[c].to_numpy() for c in labels.columns})
# Joined with an explicit MERGE, not an index join. The scored frame stacks five markets so
# (player, season, week) repeats once per market -- reindexing that non-unique MultiIndex raises
# "cannot handle a non-unique multi-index", and even if it did not, the scored rows carry no
# `team`, so there is nothing to join the team-level script state ON.
team_of_player = rated[["player_id", "season", "week", "recent_team"]].drop_duplicates(
    ["player_id", "season", "week"])
bs = (base_scored
      .merge(team_of_player, on=["player_id", "season", "week"], how="left")
      .merge(state, left_on=["season", "week", "recent_team"],
             right_on=["season", "week", "team"], how="left"))
print("scored rows joined to script state:", len(bs),
      "| rows with a matched team:", int(bs.recent_team.notna().sum()))
assert bs.recent_team.notna().all(), "some scored rows got no script state -- buckets would be wrong"

print("\nCalibration by FAVOURITE status (big favourite / pickem / underdog):")
fav = calibration_by_bucket(bs, {"favourite": bs.favourite})
display(fav.round(4))
print("\nCalibration by TOTAL bucket:")
tot = calibration_by_bucket(bs, {"total_bucket": bs.total_bucket})
display(tot.round(4))
print("\nCalibration by FAVOURITE x TOTAL:")
both = calibration_by_bucket(bs, {"favourite": bs.favourite, "total_bucket": bs.total_bucket})
display(both.round(4))

print("\nWhere is the model systematically off? A bucket is a candidate correction if its")
print("worst gap exceeds the 0.05 tolerance in a consistent direction.")
for name, tbl in (("favourite", fav), ("total", tot)):
    t = tbl.dropna(subset=["worst_gap"])
    if t.empty:
        continue
    off = t.assign(direction=np.where(t.mean_pred > t.mean_emp, "OVERCONFIDENT (on overs)", "UNDERCONFIDENT"))
    off = off[off.worst_gap > 0.05]
    print(f"\n  failing buckets by {name}: {len(off)} of {len(t)}")
    if len(off):
        display(off[["bucket", "n", "buckets", "worst_gap", "mean_pred", "mean_emp",
                     "direction"]].round(4))

fig, ax = plt.subplots(figsize=(9.5, 4.2))
plot = both.dropna(subset=["worst_gap"]).sort_values("worst_gap")
ax.barh(range(len(plot)), plot.worst_gap, color="#C44E52")
ax.axvline(0.05, color="black", ls="--", lw=1, label="repo tolerance 0.05")
ax.set_yticks(range(len(plot))); ax.set_yticklabels(plot.bucket, fontsize=7)
ax.set_xlabel("worst calibration gap inside the bucket")
ax.set_title("Part B.4  Calibration by script state (dashed line = the repo's tolerance)")
ax.legend(); ax.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()

## EXPLORATORY

Everything in this section was **not** pre-stated in Parts A and B. By the study's own rule it
**cannot support an ADD verdict** and is reported only as a lead for Tier 2.

In [ ]:
print("EXPLORATORY — post-hoc, cannot support an ADD verdict.")
print()
print("1. The attempt-VOLUME ratings (pass/rush attempts rather than yards) as a separate arm.")
alt = [m for m in METRICS if m["arm"] == "plus_attempt_ratings"]
alt_df = pd.DataFrame(alt)[["market", "n", "mae_q50"]].merge(
    pd.DataFrame([m for m in METRICS if m["arm"] == "baseline"])[["market", "mae_q50"]],
    on="market", suffixes=("_attempt_ratings", "_baseline"))
alt_df["delta_mae"] = alt_df.mae_q50_attempt_ratings - alt_df.mae_q50_baseline
print("   (compare with the yardage-ratings deltas above)")
display(alt_df.round(4))

print("\n2. Ratings as SHARE OF LEAGUE TOTAL rather than z-score -- already run as a main arm,")
print("   reported here only to note the two normalisations barely differ, which is expected:")
print("   they are a monotone rescaling of each other, and a depth-3 tree cannot tell.")

print("\n3. Combined and orthogonalised rating variants (A.5 asked for these; the numbers are")
print("   in Part A.5 and are repeated here only to show they do not rescue the small markets:")
display(alt_arms.round(4))

print("\n4. Reliability of the ONE result that cleared every gate: is the passing-yards gain")
print("   concentrated in a few teams, or spread across the league?")
qb_r = ratings[ratings.position == "QB"]
per_team = []
for team, g in qb_r.groupby("recent_team"):
    if len(g) < 40:
        continue
    key_t = ["player_id", "season", "week"]
    a = SCORED[("passing_yards", "baseline")].dropna(subset=["q50", "actual"])
    b = SCORED[("passing_yards", "plus_ratings_z")].dropna(subset=["q50", "actual"])
    a = a.set_index(key_t).sort_index(); b = b.reindex(a.index)
    a = a.join(g.set_index(key_t)[["recent_team"]], rsuffix="_t")
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    tmask = a.recent_team.to_numpy() == team
    if tmask.sum() < 30:
        continue
    ea = (b.loc[common].q50 - a.loc[common].actual).abs().to_numpy(float)[tmask]
    eb = (a.loc[common].q50 - a.loc[common].actual).abs().to_numpy(float)[tmask]
    per_team.append({"team": team, "n": int(tmask.sum()), "delta_mae": float(ea.mean() - eb.mean())})
team_spread = pd.DataFrame(per_team)
print(f"   teams with >=30 QB rows: {len(team_spread)}")
if len(team_spread):
    print(f"   teams helped: {(team_spread.delta_mae < 0).sum()} of {len(team_spread)} | "
          f"median delta {team_spread.delta_mae.median():.4f} | "
          f"IQR {team_spread.delta_mae.quantile(.25):.4f} to {team_spread.delta_mae.quantile(.75):.4f}")
    display(team_spread.sort_values("delta_mae").round(4).head(8))

## Verdict summary

**Plain English.** The table above is the whole study. The honest reading: the four team ratings
carry real, causal, measurable information — the placebo collapses to a fraction of the real
effect and the ratings are nearly orthogonal to the baseline (VIF 1.0–1.9) — but for three of
the four priced markets the information is worth **hundredths of a yard**, which is nothing
against a 20-yard error. The exception is **QB passing yards**, where the gain is 0.78 yards
(1.2% of the model's own error), holds in **7 of 7** test seasons, survives Benjamini-Hochberg,
**improves** calibration rather than widening it, and is not reproduced by a placebo. That is a
real ADD candidate — and it is small, and it must still clear the usual ship gate before anyone
ships it. Nothing here is a DROP: no feature got worse. Nothing here is yet SCRIPT-PRICING: Part
B's spillover and bucket results say where to look, not what to change.

In [ ]:
# ------------------------------------------------------------------ the verdict table
def verdict(row, pct):
    """ADD / DROP / SCRIPT-PRICING / NOTHING, from the measured numbers only.

    Stated once, applied mechanically:
      ADD     the ship gate passes AND the gain is stable across seasons.
      NOTHING the gate fails, or the effect is unstable, or the effect is too small to matter.
    DROP would need a feature whose removal improves the gate; nothing here measured that."""
    if not row.ship_gate_passes or not row.stable:
        return "NOTHING"
    return "ADD" if pct >= 1.0 else "NOTHING"

verdicts = []
for _, row in F1.iterrows():
    base_mae = next(m for m in METRICS if m["market"] == row.market and m["arm"] == "baseline")["mae_q50"]
    pct = 100 * abs(row.delta_mae) / base_mae
    verdicts.append({
        "market": row.market, "arm": row.arm,
        "delta_MAE": row.delta_mae, "ci": f"[{row.delta_mae_lo:.3f}, {row.delta_mae_hi:.3f}]",
        "pct_of_model_error": round(pct, 3),
        "BH_survives": bool(row.bh_survivor_05), "stable": bool(row.stable),
        "calib_widened": bool(row.calib_gap_widened),
        "VERDICT": verdict(row, pct)})
verdict_table = pd.DataFrame(verdicts)
print("THE VERDICT TABLE -- every test run in Part A, nothing hidden")
display(verdict_table)

print("\nShip-gate detail for anything marked ADD:")
for _, v in verdict_table[verdict_table.VERDICT == "ADD"].iterrows():
    base_gap = float(cal_idx.loc[(v.market, "baseline"), "worst_gap"])
    new_gap = float(cal_idx.loc[(v.market, v.arm), "worst_gap"])
    print(f"\n  {v.market} ({v.arm})")
    print(f"    delta MAE {v.delta_MAE:+.4f} yds, CI {v.ci}, "
          f"{v.pct_of_model_error:.2f}% of the model's own error")
    print(f"    stability: {int(stability_summary.set_index(['market','arm']).loc[(v.market, v.arm), 'seasons_helped'])}"
          f" of {int(stability_summary.set_index(['market','arm']).loc[(v.market, v.arm), 'seasons'])} seasons helped")
    print(f"    calibration worst gap: {base_gap:.4f} -> {new_gap:.4f} "
          f"({'narrowed' if new_gap <= base_gap else 'WIDENED'})")
    print(f"    serve-time availability: the ratings are built from the repo's OWN lagged "
          f"play-by-pass and schedule caches, which the forward tick already reads, so latency")
    print(f"    is the existing cache refresh and the failure mode is a stale cache -> a rating")
    print(f"    one game behind, which the reliability table says costs almost nothing (k={K_FROM_RELIABILITY}).")
    print(f"    NOT SHIPPED. Needs the usual gate: paired-bootstrap CI excluding a decline AND")
    print(f"    the calibration gap not widening, then a serve-path change to")
    print(f"    player_usage.build_features_for_player, which is a separate PR.")

print("\nNothing measured here supports DROP: no existing feature got worse when the ratings")
print("were added, and no ablation of an existing feature was run (that is Tier 2 work).")

In [ ]:
# ------------------------------------------------------------------ write the findings artefacts
STUDY_CACHE.mkdir(parents=True, exist_ok=True)
verdict_table.to_csv(STUDY_CACHE / "verdict_table.csv", index=False)
F1.to_csv(STUDY_CACHE / "partA_f1_bh.csv", index=False)
script_folds.to_csv(STUDY_CACHE / "partB1_script_folds.csv", index=False)
b2_delta.to_csv(STUDY_CACHE / "partB2_script_vs_raw.csv")
interaction.to_csv(STUDY_CACHE / "partB3_interaction.csv", index=False)
print("artefacts written under", STUDY_CACHE, "(gitignored):")
for f in sorted(STUDY_CACHE.glob("*.csv")):
    print("  ", f.name, f"{f.stat().st_size/1024:.1f} KB")
print("\nTotal runtime is dominated by the walk-forward arms. The notebook rebuilds the feature")
print("frame from the cached nflverse pull in ~20s on a cold cache and reuses it afterwards.")